# 05 — Frozen DINOv2 Embeddings

Runs DINOv2-S once over every cached slice and stores the result, so the
aggregation stack afterwards trains on cached features in minutes instead of
hours.

**Why frozen.** The competition scores a second prize track on
`AUC + RuntimeSeconds / 32400`, so inference cost is scored rather than merely
capped. A frozen trunk is cheap at submission, and it makes the aggregation
design — which is where the modelling actually is — cheap to iterate on.

**Cost:** 24,371 series × 24 slices = ~585,000 forward passes of a 22M
parameter ViT. About 25–40 min on a T4, once.

**Output:** `embeddings_NN.npy` per shard, float16, 768 dims per slice.
~0.9 GB total.

Pretrained weights are permitted — confirmed in the competition's Code
Requirements — but internet is disabled at submission, so the weights are also
saved here for the inference notebook to load from disk.

## Setup

In [ ]:
# Materialise the repo modules. Generated from source by
# tools/make_notebook.py -- do not edit here, edit src/ and regenerate.
import importlib, pathlib, sys

FILES = {
    'src/__init__.py': '',
    'src/config.py': '"""Central configuration: label names, cache geometry, and environment paths.\n\nEverything here is imported by both the training pipeline and the submission\nnotebook, so it must stay dependency-free apart from the standard library.\n"""\n\nimport os\nfrom pathlib import Path\n\n# ---------------------------------------------------------------------------\n# Labels\n# ---------------------------------------------------------------------------\n# Exact submission spelling. The apostrophe in "Baker\'s" and the spaces are\n# part of the column names -- do not normalise them anywhere that touches a\n# submission file.\nLABELS = [\n    "ACL",\n    "MCL",\n    "Medial Meniscus",\n    "Lateral Meniscus",\n    "Medial OA",\n    "Lateral OA",\n    "PF OA",\n    "Effusion",\n    "Synovitis",\n    "Baker\'s",\n    "Contusion",\n    "Fracture",\n]\n\nID_COL = "StudyInstanceUID"\nSUBMISSION_COLUMNS = [ID_COL] + LABELS\n\n# Compartment-specific labels. A horizontal flip swaps medial and lateral, so\n# any augmentation that mirrors the image must also swap these pairs.\nLATERAL_LABEL_PAIRS = [\n    ("Medial Meniscus", "Lateral Meniscus"),\n    ("Medial OA", "Lateral OA"),\n]\n\n# ---------------------------------------------------------------------------\n# Cache geometry\n# ---------------------------------------------------------------------------\n# Kept as constants so a higher-resolution variant is a config change rather\n# than a rewrite. Sizing: 224 * 224 * 24 bytes ~= 1.2 MB/series,\n# x 24,371 series ~= 30 GB, which fits Kaggle\'s 200 GB private dataset limit.\nCACHE_SIZE = 224          # in-plane resolution after resampling\nCACHE_SLICES = 24         # slices sampled per series, centred on the volume\nCACHE_DTYPE = "uint8"\n\n# Robust intensity normalisation percentiles. MRI has no standardised units,\n# so every volume is scaled against its own distribution.\nCLIP_PCT_LOW = 1.0\nCLIP_PCT_HIGH = 99.0\n\n# Canonical laterality: left knees are mirrored to match right knees so the\n# medial compartment always falls on the same side of the image.\nCANONICAL_SIDE = "R"\n\n# Mirroring is applied at load time, not baked into the cache.\n#\n# The side tag is blank on most series in this dataset and the series\n# descriptions are anonymised, so roughly 42% of series have no recoverable\n# side from metadata alone. Freezing a mirror decision into 30 GB of cache\n# would mean rebuilding all of it every time side detection improves -- and it\n# is expected to improve, via study-level propagation and then a classifier\n# trained on the series whose side is known. The cache therefore stores volumes\n# in their acquired orientation with the side recorded alongside, and the data\n# loader flips on read, which costs nothing.\nMIRROR_IN_CACHE = False\n\n# ---------------------------------------------------------------------------\n# Paths\n# ---------------------------------------------------------------------------\nCOMPETITION = "rsna-knee-abnormality-detection"\nPROJECT_ROOT = Path(__file__).resolve().parent.parent\n\n\n# What a competition data root must contain to be recognised.\n_MARKERS = ("train_series", "test_series")\n\n\ndef _detect_data_root() -> Path:\n    """Locate the competition data across Kaggle, Colab, and local runs.\n\n    An explicit KNEE_DATA_ROOT wins. Otherwise Kaggle\'s input directory is\n    searched by content rather than by name: the mount is not reliably at\n    /kaggle/input/<competition-slug>, and guessing the name silently falls\n    through to the local fallback, which then fails much later with a confusing\n    path in the message.\n    """\n    env = os.environ.get("KNEE_DATA_ROOT")\n    if env:\n        return Path(env)\n\n    kaggle_input = Path("/kaggle/input")\n    if kaggle_input.exists():\n        exact = kaggle_input / COMPETITION\n        if exact.exists():\n            return exact\n        # Search two levels: Kaggle mounts competition data under\n        # /kaggle/input/competitions/<slug>/ as well as directly under\n        # /kaggle/input/<slug>/, and which one you get is not predictable.\n        try:\n            for child in sorted(kaggle_input.iterdir()):\n                if not child.is_dir():\n                    continue\n                if any((child / m).exists() for m in _MARKERS):\n                    return child\n                for grandchild in sorted(child.iterdir()):\n                    if grandchild.is_dir() and any(\n                            (grandchild / m).exists() for m in _MARKERS):\n                        return grandchild\n        except OSError:\n            pass\n\n    return PROJECT_ROOT / "data"\n\n\nDATA_ROOT = _detect_data_root()\nMETA_DIR = PROJECT_ROOT / "data" / "meta"\nCACHE_DIR = Path(os.environ.get("KNEE_CACHE_DIR", PROJECT_ROOT / "data" / "cache"))\n\nTRAIN_CSV = META_DIR / "train.csv"\nTRAIN_SERIES_CSV = META_DIR / "train_series.csv"\nTEST_CSV = META_DIR / "test.csv"\nTEST_SERIES_CSV = META_DIR / "test_series.csv"\nSAMPLE_SUBMISSION_CSV = META_DIR / "sample_submission.csv"\n\nINDEX_PARQUET = META_DIR / "index.parquet"\n\nON_KAGGLE = Path("/kaggle").exists()\n',
    'src/data/__init__.py': '',
    'src/data/dicom_io.py': '"""DICOM reading, geometric slice ordering, and laterality resolution.\n\nTwo operations here are load-bearing and fail silently if done naively:\n\n1. **Slice ordering.** InstanceNumber is not reliable across the 19 contributing\n   institutions. Slices are ordered by projecting ImagePositionPatient onto the\n   slice normal derived from ImageOrientationPatient, which is defined by the\n   scanner geometry rather than by convention.\n\n2. **Laterality.** Four of the twelve labels are compartment-specific\n   (medial vs. lateral). In an image of a left knee the medial compartment sits\n   on the opposite side of the frame from a right knee, so left knees must be\n   mirrored into a canonical orientation before a model can learn the\n   distinction.\n"""\n\nimport re\nfrom pathlib import Path\nfrom typing import Iterable, List, Optional, Sequence\n\nimport numpy as np\nimport pydicom\n\n# Header fields pulled during the index scan. Kept explicit so the parquet\n# schema is stable and the scan stays header-only.\nHEADER_FIELDS = [\n    "StudyInstanceUID",\n    "SeriesInstanceUID",\n    "SOPInstanceUID",\n    "Rows",\n    "Columns",\n    "PixelSpacing",\n    "ImagePositionPatient",\n    "ImageOrientationPatient",\n    "SliceThickness",\n    "SpacingBetweenSlices",\n    "EchoTime",\n    "RepetitionTime",\n    "InversionTime",\n    "ScanningSequence",\n    "SequenceVariant",\n    # (0018,0010). The one header that states contrast administration\n    # outright, and the index built before 2026-08-31 does not carry it --\n    # so the contrast audit has to fall back on typed descriptions, 17.7% of\n    # which are anonymised away. Absence of a string is not absence of\n    # contrast; absence of this tag is at least a direct question asked.\n    "ContrastBolusAgent",\n    "SeriesDescription",\n    "ProtocolName",\n    "BodyPartExamined",\n    "Manufacturer",\n    "ManufacturerModelName",\n    "MagneticFieldStrength",\n    "ImageLaterality",\n    "Laterality",\n    "InstanceNumber",\n]\n\n# DICOM DS/IS values carry their original string representation, which would\n# land in the index as object dtype. Cast these explicitly.\nNUMERIC_FIELDS = {\n    "Rows",\n    "Columns",\n    "SliceThickness",\n    "SpacingBetweenSlices",\n    "EchoTime",\n    "RepetitionTime",\n    "InversionTime",\n    "MagneticFieldStrength",\n    "InstanceNumber",\n}\n\n_LEFT_PATTERN = re.compile(\n    r"\\b(l|lt|left|links|izquierda|izq|gauche|sinistra|sin|links?e)\\b",\n    re.IGNORECASE,\n)\n_RIGHT_PATTERN = re.compile(\n    r"\\b(r|rt|right|rechts|derecha|der|droite|destra|dx)\\b",\n    re.IGNORECASE,\n)\n\n\ndef read_header(path) -> dict:\n    """Read one DICOM\'s headers without decoding pixel data."""\n    ds = pydicom.dcmread(str(path), stop_before_pixels=True, force=True)\n    row = {"path": str(path)}\n    for field in HEADER_FIELDS:\n        value = getattr(ds, field, None)\n        if value is None:\n            row[field] = None\n        elif isinstance(value, pydicom.multival.MultiValue):\n            # Geometry fields are numeric; ScanningSequence and friends are\n            # multi-valued strings, so fall back to a joined token.\n            try:\n                row[field] = [float(v) for v in value]\n            except (TypeError, ValueError):\n                row[field] = "\\\\".join(str(v) for v in value)\n        elif field in NUMERIC_FIELDS:\n            try:\n                row[field] = float(value)\n            except (TypeError, ValueError):\n                row[field] = None\n        elif isinstance(value, (int, float, str)):\n            row[field] = str(value) if isinstance(value, str) else value\n        else:\n            row[field] = str(value)\n    return row\n\n\ndef slice_normal(orientation: Optional[Sequence[float]]) -> np.ndarray:\n    """Unit normal of the image plane, from the two direction cosines.\n\n    Falls back to the z axis when orientation is missing, which keeps a\n    malformed series ordered by patient-superior position rather than crashing.\n    """\n    if orientation is None or len(orientation) != 6:\n        return np.array([0.0, 0.0, 1.0])\n\n    row_cosine = np.asarray(orientation[:3], dtype=float)\n    col_cosine = np.asarray(orientation[3:], dtype=float)\n    normal = np.cross(row_cosine, col_cosine)\n\n    magnitude = np.linalg.norm(normal)\n    if magnitude == 0:\n        return np.array([0.0, 0.0, 1.0])\n    return normal / magnitude\n\n\ndef slice_position(header: dict) -> float:\n    """Scalar position of a slice along its own stack axis."""\n    position = header.get("ImagePositionPatient")\n    if position is None or len(position) != 3:\n        # No geometry available; fall back to InstanceNumber so ordering is at\n        # least deterministic. The index scan flags these series separately.\n        number = header.get("InstanceNumber")\n        return float(number) if number is not None else 0.0\n\n    normal = slice_normal(header.get("ImageOrientationPatient"))\n    return float(np.dot(np.asarray(position, dtype=float), normal))\n\n\ndef sort_slices(headers: Iterable[dict]) -> List[dict]:\n    """Order a series\' slices along the acquisition axis."""\n    return sorted(headers, key=slice_position)\n\n\ndef has_geometry(headers: Sequence[dict]) -> bool:\n    """True when every slice carries the position and orientation tags."""\n    for header in headers:\n        position = header.get("ImagePositionPatient")\n        orientation = header.get("ImageOrientationPatient")\n        if position is None or len(position) != 3:\n            return False\n        if orientation is None or len(orientation) != 6:\n            return False\n    return True\n\n\ndef resolve_laterality(headers: Sequence[dict]) -> Optional[str]:\n    """Determine whether a series images a left or right knee.\n\n    Prefers the explicit DICOM tags, then falls back to parsing the free-text\n    description fields, which carry side markers in several languages. Returns\n    "L", "R", or None when the side cannot be established -- callers must treat\n    None as unusable rather than guessing, since a wrong side silently swaps the\n    medial and lateral labels.\n    """\n    for header in headers:\n        for field in ("ImageLaterality", "Laterality"):\n            value = header.get(field)\n            if isinstance(value, str):\n                # The tag holds "L"/"R" but also "LEFT"/"RIGHT", and is far more\n                # often present-but-empty than absent. An empty string is not a\n                # side and must not be read as one.\n                token = value.strip().upper()\n                if token in ("L", "LEFT"):\n                    return "L"\n                if token in ("R", "RIGHT"):\n                    return "R"\n\n    for field in ("SeriesDescription", "ProtocolName", "BodyPartExamined"):\n        for header in headers:\n            text = header.get(field)\n            if not isinstance(text, str):\n                continue\n            # Split on non-letters so "MR_Knie_Rechts" yields usable tokens.\n            spaced = re.sub(r"[^A-Za-zÀ-ÿ]+", " ", text)\n            left = bool(_LEFT_PATTERN.search(spaced))\n            right = bool(_RIGHT_PATTERN.search(spaced))\n            # Ambiguous strings mentioning both sides tell us nothing.\n            if left and not right:\n                return "L"\n            if right and not left:\n                return "R"\n\n    return None\n\n\ndef load_series_volume(paths: Sequence[str]) -> np.ndarray:\n    """Decode an ordered list of DICOM paths into a (slices, rows, cols) array.\n\n    Applies the rescale slope and intercept when present. Slices whose in-plane\n    shape differs from the first are skipped rather than silently broadcast.\n    """\n    slices = []\n    reference_shape = None\n    skipped = 0\n\n    for path in paths:\n        try:\n            ds = pydicom.dcmread(str(path), force=True)\n            pixels = ds.pixel_array.astype(np.float32)\n        except Exception:  # noqa: BLE001\n            # A truncated or malformed file loses one slice, not the series.\n            # The dataset contains files whose declared pixel length exceeds\n            # what is actually stored, and dropping 23 good slices over one\n            # bad one costs a whole study its imaging.\n            skipped += 1\n            continue\n\n        slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)\n        intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)\n        if slope != 1.0 or intercept != 0.0:\n            pixels = pixels * slope + intercept\n\n        if reference_shape is None:\n            reference_shape = pixels.shape\n        elif pixels.shape != reference_shape:\n            skipped += 1\n            continue\n\n        slices.append(pixels)\n\n    if not slices:\n        raise ValueError(f"no decodable slices in series ({skipped} unreadable)")\n\n    return np.stack(slices, axis=0)\n\n\ndef series_paths(series_dir) -> List[str]:\n    """All DICOM files under one series directory."""\n    return sorted(str(p) for p in Path(series_dir).glob("*.dcm"))\n\n\ndef resolve_study_laterality(series_sides: dict) -> dict:\n    """Fill in unknown series sides from other series in the same study.\n\n    A knee MRI study images one knee, so a side established on any series\n    applies to all of them. This matters because the DICOM side tags are\n    populated inconsistently across the contributing institutions -- a study\n    can carry the tag on its coronal series and omit it on the sagittals.\n\n    Report text is deliberately not used as a source here. Reports exist only\n    for training studies, so recovering a side from them would mirror the\n    training volumes and leave the test volumes untouched -- a train/test skew\n    that is worse than the missing side it fixes.\n\n    Args:\n        series_sides: {study_uid: {series_uid: "L" | "R" | None}}\n\n    Returns:\n        The same structure with unknown entries filled where the study allows.\n        Studies whose series disagree are left untouched, since a conflict\n        means the assumption of one knee per study does not hold there.\n    """\n    resolved = {}\n    for study_uid, series in series_sides.items():\n        known = {side for side in series.values() if side in ("L", "R")}\n\n        if len(known) == 1:\n            side = known.pop()\n            resolved[study_uid] = {uid: side for uid in series}\n        else:\n            # No evidence, or contradictory evidence. Guessing would silently\n            # swap the medial and lateral labels on four of twelve findings.\n            resolved[study_uid] = dict(series)\n\n    return resolved\n\n\ndef laterality_coverage(series_sides: dict) -> dict:\n    """Counts before and after study-level propagation, for auditing."""\n    before = sum(1 for s in series_sides.values() for v in s.values()\n                 if v in ("L", "R"))\n    total = sum(len(s) for s in series_sides.values())\n\n    filled = resolve_study_laterality(series_sides)\n    after = sum(1 for s in filled.values() for v in s.values()\n                if v in ("L", "R"))\n\n    conflicts = sum(\n        1 for s in series_sides.values()\n        if len({v for v in s.values() if v in ("L", "R")}) > 1\n    )\n\n    return {\n        "total_series": total,\n        "known_before": before,\n        "known_after": after,\n        "recovered": after - before,\n        "conflicting_studies": conflicts,\n    }\n',
    'src/data/cache.py': '"""Turn a raw DICOM series into a compact, model-ready uint8 volume.\n\nThe same code path runs during cache construction and at submission time. Any\ndivergence between the two produces a train/test preprocessing skew, which is\ninvisible in local validation and only shows up as a collapsed leaderboard\nscore -- so callers must import from here rather than reimplementing.\n\nOutput per series: (CACHE_SLICES, CACHE_SIZE, CACHE_SIZE) uint8, in the\norientation it was acquired in. Mirroring to a canonical side happens at load\ntime rather than here -- see config.MIRROR_IN_CACHE for why.\n"""\n\nfrom typing import Dict, List, Optional, Sequence\n\nimport cv2\nimport numpy as np\n\nfrom src import config\nfrom src.data import dicom_io\n\n\ndef normalize_intensity(volume: np.ndarray) -> np.ndarray:\n    """Scale a volume to uint8 using its own robust percentile range.\n\n    MRI carries no standardised units -- intensity depends on scanner, field\n    strength, coil, and whether fat suppression was applied -- so each volume is\n    scaled against its own distribution rather than a global constant.\n    """\n    finite = volume[np.isfinite(volume)]\n    if finite.size == 0:\n        return np.zeros(volume.shape, dtype=np.uint8)\n\n    low, high = np.percentile(finite, [config.CLIP_PCT_LOW, config.CLIP_PCT_HIGH])\n    if not np.isfinite(low) or not np.isfinite(high) or high <= low:\n        # Flat or degenerate volume; nothing to stretch.\n        return np.zeros(volume.shape, dtype=np.uint8)\n\n    scaled = (np.clip(volume, low, high) - low) / (high - low)\n    return (scaled * 255.0).round().astype(np.uint8)\n\n\ndef resize_volume(volume: np.ndarray, size: int = None) -> np.ndarray:\n    """Resample every slice to size x size with area/linear interpolation."""\n    size = size or config.CACHE_SIZE\n    output = np.empty((volume.shape[0], size, size), dtype=volume.dtype)\n\n    for i, plane in enumerate(volume):\n        # Area interpolation avoids the aliasing that bilinear introduces when\n        # downsampling 800px acquisitions; linear is correct when upsampling.\n        interp = cv2.INTER_AREA if plane.shape[0] > size else cv2.INTER_LINEAR\n        output[i] = cv2.resize(plane, (size, size), interpolation=interp)\n\n    return output\n\n\ndef select_slices(count: int, target: int = None) -> List[int]:\n    """Indices of `target` slices spread evenly across a stack of `count`.\n\n    Short series are handled by repeating edge slices rather than zero-padding,\n    so the volume stays anatomically continuous.\n    """\n    target = target or config.CACHE_SLICES\n    if count <= 0:\n        raise ValueError("cannot select slices from an empty series")\n    if count == target:\n        return list(range(count))\n\n    positions = np.linspace(0, count - 1, target)\n    return [int(round(p)) for p in positions]\n\n\ndef mirror_axis(orientation: Optional[Sequence[float]]) -> int:\n    """Which array axis runs medial-to-lateral, from the scanner geometry.\n\n    Mirroring a knee means reflecting through the patient\'s left-right axis,\n    and which array axis that corresponds to depends on the acquisition plane:\n\n        coronal / axial   left-right lies in-plane  -> flip columns (axis 2)\n        sagittal          left-right is the slice axis -> reverse slices (axis 0)\n\n    A blanket column flip is wrong for sagittal series. It leaves medial and\n    lateral exactly where they were -- they are separated across slices, not\n    across the image -- and instead reflects anterior-posterior, moving the\n    patella to the back of the knee.\n\n    Returns 0, 1, or 2: the axis of a (slices, rows, cols) volume to reverse.\n    """\n    left_right = np.array([1.0, 0.0, 0.0])\n\n    if orientation is None or len(orientation) != 6:\n        # No geometry to reason from. Columns is the common case (coronal and\n        # axial together outnumber sagittal), but the caller should prefer not\n        # to mirror at all when geometry is missing.\n        return 2\n\n    width = np.asarray(orientation[:3], dtype=float)    # increasing column index\n    height = np.asarray(orientation[3:], dtype=float)   # increasing row index\n    normal = dicom_io.slice_normal(orientation)\n\n    alignments = [\n        (abs(float(normal @ left_right)), 0),\n        (abs(float(height @ left_right)), 1),\n        (abs(float(width @ left_right)), 2),\n    ]\n    return max(alignments)[1]\n\n\ndef canonicalize_side(volume: np.ndarray, side: Optional[str],\n                      orientation: Optional[Sequence[float]] = None) -> np.ndarray:\n    """Mirror left knees so the medial compartment is always on one side.\n\n    Four labels are compartment-specific (Medial/Lateral Meniscus, Medial/\n    Lateral OA). Without this the model would have to infer the side before it\n    could place a finding, from a cue that is not reliably in the image.\n\n    The axis reflected is chosen from the acquisition geometry -- see\n    mirror_axis -- because it differs between sagittal and coronal series.\n    """\n    if side is None:\n        # Unknown side: leave untouched and let the caller decide. Guessing\n        # would silently swap four of the twelve labels.\n        return volume\n    if side.upper() == config.CANONICAL_SIDE:\n        return volume\n\n    axis = mirror_axis(orientation)\n    return np.flip(volume, axis=axis).copy()\n\n\ndef submission_indices(count: int, slices: int) -> List[int]:\n    """Raw slice indices a `slices`-deep submission actually reads.\n\n    **Not** `select_slices(count, slices)`, and the difference is the whole\n    reason this exists. The submission decodes `CACHE_SLICES` and then\n    subsamples with `np.linspace(0, CACHE_SLICES - 1, TUNED_SLICES)`; picking\n    `slices` evenly across the raw stack in one step lands on *different\n    files* for 41 of the 44 series lengths between 16 and 60. The model would\n    then be scored on anatomy it was not trained to read, and nothing would\n    raise -- the shapes agree either way. That is the same failure the\n    `volumes[:MAX_SERIES]` head truncation already cost this project.\n\n    So the two selections are composed rather than collapsed: the coarse pick\n    the cache would have made, then the fine pick the model applies to it.\n    """\n    coarse = select_slices(count)\n    if slices == config.CACHE_SLICES:\n        return coarse\n    return [coarse[i] for i in select_slices(config.CACHE_SLICES, slices)]\n\n\ndef build_series_volume(headers: Sequence[dict],\n                        slices: Optional[int] = None) -> Dict:\n    """Full pipeline for one series: sort, decode, normalise, resize.\n\n    Mirroring is applied only when config.MIRROR_IN_CACHE is set, which it is\n    not by default. The recorded `laterality` is what the loader mirrors on.\n    Returns the volume plus the provenance needed to audit it later.\n\n    `slices` exists for the submission, which reads only `TUNED_SLICES` of the\n    `CACHE_SLICES` this decodes: at 16 of 24 that is a third of the pixel\n    decoding done for images the model never sees, and inference time is a\n    *scored quantity* in the Efficiency Prize. It defaults to None, so cache\n    construction is unchanged and every existing shard stays valid.\n\n    **It is not bit-identical to decoding the full depth.**\n    `normalize_intensity` scales each volume by its own 1st/99th percentiles,\n    computed over whatever was decoded, so a shallower stack shifts the\n    intensity mapping slightly. The slices themselves are the same files --\n    see `submission_indices` -- but the uint8 values differ a little, and the\n    submission notebook measures that difference rather than assuming it away.\n    """\n    if not headers:\n        raise ValueError("no headers supplied")\n\n    depth = config.CACHE_SLICES if slices is None else slices\n    ordered = dicom_io.sort_slices(headers)\n    first_header = ordered[0]\n    side = dicom_io.resolve_laterality(ordered)\n    geometry_ok = dicom_io.has_geometry(ordered)\n\n    indices = submission_indices(len(ordered), depth)\n    selected = [ordered[i] for i in indices]\n\n    raw = dicom_io.load_series_volume([h["path"] for h in selected])\n\n    # Slices skipped as unreadable leave the volume short of the requested\n    # depth, and the cache is a fixed-shape array. Re-select across whatever\n    # decoded so the depth is exact regardless of how many files were dropped.\n    if raw.shape[0] != depth:\n        raw = raw[select_slices(raw.shape[0], depth)]\n\n    resized = resize_volume(raw.astype(np.float32))\n    normalized = normalize_intensity(resized)\n\n    # Left as acquired by default; the loader mirrors using the recorded side.\n    # See config.MIRROR_IN_CACHE for why this is not baked in here.\n    volume = (canonicalize_side(normalized, side,\n                                first_header.get("ImageOrientationPatient"))\n              if config.MIRROR_IN_CACHE else normalized)\n\n    first = ordered[0]\n    return {\n        "volume": volume,\n        "StudyInstanceUID": first.get("StudyInstanceUID"),\n        "SeriesInstanceUID": first.get("SeriesInstanceUID"),\n        "n_slices_raw": len(ordered),\n        "laterality": side,\n        "mirrored": bool(config.MIRROR_IN_CACHE and side is not None\n                         and side.upper() != config.CANONICAL_SIDE),\n        "geometry_ok": geometry_ok,\n        "rows": int(first.get("Rows") or 0),\n        "cols": int(first.get("Columns") or 0),\n    }\n\n\ndef build_series_record(item):\n    """Picklable worker for a process pool: (series_uid, headers) -> result.\n\n    Defined at module level deliberately. A worker defined inside a notebook\n    cell cannot be pickled by name, so ProcessPoolExecutor fails under the\n    spawn start method and survives under fork only by accident. Keeping it\n    importable makes the cache build work the same way everywhere.\n\n    Failures are returned rather than raised: one unreadable series must not\n    end a ninety-minute run, and the manifest records what to investigate.\n    """\n    series_uid, headers = item\n    try:\n        result = build_series_volume(headers)\n        return series_uid, result["volume"], {\n            "SeriesInstanceUID": series_uid,\n            "StudyInstanceUID": result["StudyInstanceUID"],\n            "n_slices_raw": result["n_slices_raw"],\n            "laterality": result["laterality"],\n            "mirrored": result["mirrored"],\n            "geometry_ok": result["geometry_ok"],\n            "rows": result["rows"],\n            "cols": result["cols"],\n            "error": None,\n        }\n    except Exception as exc:  # noqa: BLE001 - recorded, not raised\n        study_uid = headers[0].get("StudyInstanceUID") if headers else None\n        return series_uid, None, {\n            "SeriesInstanceUID": series_uid,\n            "StudyInstanceUID": study_uid,\n            "n_slices_raw": len(headers) if headers else 0,\n            "laterality": None,\n            "mirrored": False,\n            "geometry_ok": False,\n            "rows": 0,\n            "cols": 0,\n            "error": str(exc)[:200],\n        }\n',
    'src/data/index.py': '"""Header-only scan of the DICOM tree into a single parquet index.\n\nRuns on Kaggle, where the competition data is mounted. Reads no pixel data, so\nthe cost is dominated by file-system traversal rather than decoding. The\nresulting index drives cache construction and lets series be filtered and\naudited without touching the 570 GB of imaging again.\n\nUsage (in a Kaggle notebook):\n\n    from src.data.index import build_index\n    build_index("train_series", out_path="/kaggle/working/index_train.parquet")\n"""\n\nimport argparse\nimport os\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\nfrom typing import Iterator, List, Optional\n\nimport pandas as pd\n\nfrom src import config\nfrom src.data import dicom_io\n\n\ndef iter_dicom_paths(root) -> Iterator[Path]:\n    """Yield every .dcm under root/study/series/ without materialising a list."""\n    root = Path(root)\n    for study_dir in sorted(root.iterdir()):\n        if not study_dir.is_dir():\n            continue\n        for series_dir in sorted(study_dir.iterdir()):\n            if not series_dir.is_dir():\n                continue\n            for path in sorted(series_dir.glob("*.dcm")):\n                yield path\n\n\ndef scan_paths(paths: List[Path], workers: int = 8) -> pd.DataFrame:\n    """Read headers for a batch of files in parallel.\n\n    Header reads are I/O bound, so threads are the right tool -- the GIL is\n    released during file reads and a process pool would only add overhead.\n    """\n    def safe_read(path):\n        try:\n            return dicom_io.read_header(path)\n        except Exception as exc:  # noqa: BLE001 - one bad file must not stop a scan\n            return {"path": str(path), "error": str(exc)}\n\n    with ThreadPoolExecutor(max_workers=workers) as pool:\n        rows = list(pool.map(safe_read, paths))\n\n    return pd.DataFrame(rows)\n\n\ndef build_index(\n    split: str = "train_series",\n    out_path: Optional[str] = None,\n    data_root=None,\n    workers: int = 8,\n    batch_size: int = 5000,\n    limit: Optional[int] = None,\n) -> pd.DataFrame:\n    """Scan one split\'s DICOM tree and write a parquet index.\n\n    Processes in batches so a long scan reports progress and so peak memory\n    stays flat regardless of how many files the split holds.\n    """\n    root = Path(data_root or config.DATA_ROOT) / split\n    if not root.exists():\n        raise FileNotFoundError(f"{root} does not exist")\n\n    try:\n        from tqdm.auto import tqdm\n    except ImportError:  # progress is a convenience, never a requirement\n        tqdm = None\n\n    frames = []\n    batch: List[Path] = []\n    scanned = 0\n\n    # The tree is walked lazily, so the total is unknown until the end. tqdm\n    # counts files without a total rather than forcing a full listing first,\n    # which on this dataset would mean stat-ing ~820,000 paths up front.\n    progress = tqdm(total=limit, unit="file", desc=f"scanning {split}") if tqdm else None\n\n    for path in iter_dicom_paths(root):\n        batch.append(path)\n        if limit is not None and len(batch) + scanned >= limit:\n            break\n        if len(batch) >= batch_size:\n            frames.append(scan_paths(batch, workers=workers))\n            scanned += len(batch)\n            if progress is not None:\n                progress.update(len(batch))\n                progress.set_postfix(series=int(frames[-1]["SeriesInstanceUID"].nunique()))\n            else:\n                print(f"  scanned {scanned} files", flush=True)\n            batch = []\n\n    if batch:\n        frames.append(scan_paths(batch, workers=workers))\n        scanned += len(batch)\n        if progress is not None:\n            progress.update(len(batch))\n\n    if progress is not None:\n        progress.close()\n\n    if not frames:\n        raise RuntimeError(f"no DICOM files found under {root}")\n\n    index = pd.concat(frames, ignore_index=True)\n    print(f"scanned {scanned} files across {index[\'SeriesInstanceUID\'].nunique()} series")\n\n    if "error" in index.columns:\n        failures = int(index["error"].notna().sum())\n        if failures:\n            print(f"WARNING: {failures} files failed to parse")\n\n    if out_path:\n        os.makedirs(Path(out_path).parent, exist_ok=True)\n        index.to_parquet(out_path, index=False)\n        print(f"wrote {out_path}")\n\n    return index\n\n\ndef summarize(index: pd.DataFrame) -> pd.DataFrame:\n    """Per-series roll-up: slice count, geometry completeness, laterality.\n\n    Surfaces the series that need attention -- missing geometry, unresolvable\n    side -- before an hour of cache construction is spent on them.\n    """\n    records = []\n    for series_uid, group in index.groupby("SeriesInstanceUID", sort=False):\n        headers = group.to_dict("records")\n        records.append({\n            "StudyInstanceUID": headers[0].get("StudyInstanceUID"),\n            "SeriesInstanceUID": series_uid,\n            "n_slices": len(headers),\n            "geometry_ok": dicom_io.has_geometry(headers),\n            "laterality": dicom_io.resolve_laterality(headers),\n            "rows": headers[0].get("Rows"),\n            "cols": headers[0].get("Columns"),\n        })\n\n    return pd.DataFrame(records)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--split", default="train_series")\n    parser.add_argument("--out", default=None)\n    parser.add_argument("--workers", type=int, default=8)\n    parser.add_argument("--limit", type=int, default=None)\n    args = parser.parse_args()\n\n    build_index(split=args.split, out_path=args.out,\n                workers=args.workers, limit=args.limit)\n\n\nif __name__ == "__main__":\n    main()\n',
    'src/model/__init__.py': '',
    'src/model/side_classifier.py': '"""Predict whether a cached series images a left or right knee.\n\nThe DICOM side tag is blank on ~41% of series and the descriptions are\nanonymised, so metadata cannot resolve them. Four of the twelve findings are\ncompartment-specific, and placing them requires knowing the side, so the\nremainder is predicted from the imaging itself.\n\nThis trains on the 14,403 series whose side is known and applies to the rest.\nBoth halves come from the same cache, so it behaves identically at submission\ntime -- unlike report text, which exists only for training studies.\n\n**Why the whole ordered stack rather than one representative slice.** For\ncoronal and axial acquisitions the left-right axis lies in-plane, and a single\nimage settles the side: the fibula sits laterally. For sagittal acquisitions\nthe left-right axis IS the slice axis, so individual slices look much the same\neither way -- what differs is the direction of travel through the volume, which\nruns medial-to-lateral for one side and lateral-to-medial for the other. A\nmodel shown one slice cannot see that. Slices are therefore fed as ordered\nchannels, which preserves it.\n"""\n\nfrom typing import Optional, Sequence\n\nimport numpy as np\n\ntry:\n    import torch\n    import torch.nn as nn\nexcept ImportError:  # the module is importable without torch for testing\n    torch = None\n    nn = object\n\nfrom src import config\n\n# Downsampled in-plane resolution. The side of a knee is a coarse, global cue,\n# so half resolution is ample and quadruples throughput.\nINPUT_SIZE = 112\n\nSIDES = ["L", "R"]\n\n\ndef prepare_batch(volumes: np.ndarray, size: int = INPUT_SIZE) -> np.ndarray:\n    """Cache volumes to model input: (N, slices, size, size) float32 in [0, 1].\n\n    Slices become channels, keeping their order, so the network can use the\n    medial-to-lateral direction that distinguishes sides on sagittal series.\n    """\n    import cv2\n\n    if volumes.ndim == 3:\n        volumes = volumes[None]\n\n    count, depth, _, _ = volumes.shape\n    output = np.empty((count, depth, size, size), dtype=np.float32)\n\n    for i in range(count):\n        for j in range(depth):\n            output[i, j] = cv2.resize(volumes[i, j], (size, size),\n                                      interpolation=cv2.INTER_AREA)\n\n    return output / 255.0\n\n\ndef flip_input(batch: np.ndarray, plane_axis: Sequence[int]) -> np.ndarray:\n    """Mirror a prepared batch along each item\'s medial-lateral axis.\n\n    Used for augmentation: mirroring an input flips its label, which doubles\n    the training set and forces the model to key on anatomy rather than on any\n    incidental left/right imbalance in the source institutions.\n\n    `plane_axis` gives the cache axis (0 for slices, 2 for columns) per item;\n    in the prepared batch slices are dimension 1 and columns dimension 3.\n    """\n    output = np.empty_like(batch)\n    for i, axis in enumerate(plane_axis):\n        output[i] = np.flip(batch[i], axis=0 if axis == 0 else 2)\n    return output\n\n\nif torch is None:\n    # Importable without torch so the array handling above can be tested on a\n    # machine that never trains. The network itself needs the real thing.\n    SideNet = None\nelse:\n  class SideNet(nn.Module):\n      """A small convolutional classifier over slice-channel input.\n\n      Trained from scratch deliberately. It is a coarse two-class problem with\n      14,000 labelled examples, so a pretrained backbone buys little -- and the\n      competition\'s stance on pretrained weights is still unconfirmed, which this\n      piece of the pipeline should not depend on.\n      """\n\n      def __init__(self, in_channels: int = None, width: int = 32):\n          super().__init__()\n          in_channels = in_channels or config.CACHE_SLICES\n\n          def block(inp, out, stride=2):\n              return nn.Sequential(\n                  nn.Conv2d(inp, out, 3, stride=stride, padding=1, bias=False),\n                  nn.BatchNorm2d(out),\n                  nn.ReLU(inplace=True),\n                  nn.Conv2d(out, out, 3, padding=1, bias=False),\n                  nn.BatchNorm2d(out),\n                  nn.ReLU(inplace=True),\n              )\n\n          self.features = nn.Sequential(\n              block(in_channels, width),        # 112 -> 56\n              block(width, width * 2),          # 56 -> 28\n              block(width * 2, width * 4),      # 28 -> 14\n              block(width * 4, width * 4),      # 14 -> 7\n          )\n          self.head = nn.Sequential(\n              nn.AdaptiveAvgPool2d(1),\n              nn.Flatten(),\n              nn.Dropout(0.2),\n              nn.Linear(width * 4, 1),\n          )\n\n      def forward(self, x):\n          return self.head(self.features(x)).squeeze(-1)\n\n\ndef study_level_vote(predictions, study_uids, confidences=None) -> dict:\n    """Collapse per-series predictions to one side per study.\n\n    A study is one knee, so its series must agree. Voting across them is\n    strictly more reliable than any single series -- and the observed\n    conflicting-tag rate is under 1%, so the assumption holds in practice.\n\n    Returns {study_uid: "L" | "R"}.\n    """\n    if len(predictions) != len(study_uids):\n        raise ValueError(\n            f"predictions and study_uids differ in length: "\n            f"{len(predictions)} vs {len(study_uids)}")\n    if confidences is None:\n        confidences = np.ones(len(predictions))\n    elif len(confidences) != len(predictions):\n        # zip would truncate silently, shifting weights onto the wrong series.\n        raise ValueError(\n            f"confidences length {len(confidences)} does not match "\n            f"{len(predictions)} predictions")\n\n    scores = {}\n    for prediction, study, weight in zip(predictions, study_uids, confidences):\n        scores.setdefault(study, []).append((float(prediction), float(weight)))\n\n    result = {}\n    for study, entries in scores.items():\n        total = sum(weight for _, weight in entries)\n        mean = (sum(p * w for p, w in entries) / total) if total else 0.5\n        result[study] = "R" if mean >= 0.5 else "L"\n    return result\n\n\ndef accuracy(predictions: np.ndarray, truth: np.ndarray) -> float:\n    """Fraction correct, with predictions as probabilities of the right knee."""\n    return float(((predictions >= 0.5).astype(int) == truth).mean())\n',
    'src/model/embed.py': '"""Frozen DINOv2 slice embeddings.\n\nRuns the backbone once over every cached slice and stores the result, so the\naggregation stack trains on cached features in minutes rather than hours. The\nsame code runs at submission time, which is the point -- a divergence between\nhow features are computed for training and for inference is invisible locally\nand fatal on the leaderboard.\n\n**Why frozen, and why first.** The competition scores a second prize track on\n`AUC + RuntimeSeconds / 32400`, so inference cost is a scored quantity rather\nthan a ceiling to stay under. A frozen trunk is cheap at submission and lets\nthe aggregation design -- which is where the modelling actually happens -- be\niterated dozens of times a day instead of three.\n\n**What is stored per slice.** The CLS token concatenated with the patch tokens\nreduced by `POOL_GRID`:\n\n    grid 1     CLS + mean of all patches      768 dims    0.9 GB\n    grid 2     CLS + 2x2 pooled grid        1,920 dims    2.3 GB\n    grid 4     CLS + 4x4 pooled grid        6,528 dims    7.6 GB\n\nStoring the full 16x16 grid would cost ~115 GB, which is why it is not offered.\n\nThe first model used grid 1, and its weakness appeared exactly where the mean\npredicts one: regional findings (effusion, OA, Baker\'s) averaged 0.865 on gold\nwhile focal ones (meniscus, ligaments, fracture) averaged 0.695. A meniscal\ntear occupies two or three patches out of 256, so a mean over all of them\nburies it in 250-odd patches of normal tissue.\n\nWide embeddings must be projected before the aggregation stack -- see\n`KneeNet(input_dim=...)`. Running the stack at 6,528 wide puts ~769M parameters\nin it against 8.2M with a projection.\n"""\n\nfrom typing import Optional\n\nimport numpy as np\n\ntry:\n    import torch\n    import torch.nn as nn\nexcept ImportError:\n    torch = None\n    nn = None\n\nfrom src import config\n\n# The registers variant. Registers exist to absorb the high-norm artefact\n# tokens that otherwise contaminate patch features -- precisely what gets\n# pooled here.\nMODEL_NAME = "vit_small_patch14_reg4_dinov2.lvd142m"\n\n# timm\'s DINOv2 defaults to 518. The cache is 224, which divides cleanly by the\n# patch size of 14 into a 16x16 grid, so the size is passed explicitly and the\n# position embeddings are interpolated rather than silently mismatched.\nINPUT_SIZE = config.CACHE_SIZE\n\n# DINOv2-S token width.\nTOKEN_DIM = 384\n\n# How the 16x16 patch grid is reduced before storage.\n#\n#   1  mean over all patches       768 dims   0.9 GB over the dataset\n#   4  4x4 average-pooled grid   6,528 dims   7.6 GB\n#\n# The mean is what the first model used, and its weakness showed up exactly\n# where predicted: regional findings averaged 0.865 on gold while focal ones\n# averaged 0.695. A meniscal tear occupies two or three patches out of 256, so\n# averaging buries it. A 4x4 grid keeps enough spatial structure for the\n# attention pooling to find it.\nPOOL_GRID = 1\n\n\ndef embed_dim(grid: int = None) -> int:\n    """Stored width per slice: CLS plus the pooled patch tokens."""\n    grid = POOL_GRID if grid is None else grid\n    tokens = 1 + (grid * grid if grid > 1 else 1)\n    return TOKEN_DIM * tokens\n\n\nEMBED_DIM = embed_dim()\n\n# ImageNet statistics, which DINOv2 was trained under.\nMEAN = (0.485, 0.456, 0.406)\nSTD = (0.229, 0.224, 0.225)\n\n\ndef build_encoder(model_name: str = MODEL_NAME, pretrained: bool = True,\n                  checkpoint_path: Optional[str] = None):\n    """A frozen DINOv2 feature extractor in eval mode.\n\n    `checkpoint_path` loads weights from disk, which the submission notebook\n    needs: internet is disabled there, so timm cannot fetch anything.\n    """\n    import timm\n\n    model = timm.create_model(\n        model_name,\n        pretrained=pretrained and checkpoint_path is None,\n        img_size=INPUT_SIZE,\n        num_classes=0,\n    )\n\n    if checkpoint_path:\n        state = torch.load(checkpoint_path, map_location="cpu")\n        state = state.get("state_dict", state)\n        model.load_state_dict(state, strict=False)\n\n    model.eval()\n    for parameter in model.parameters():\n        parameter.requires_grad = False\n    return model\n\n\ndef normalize(slices):\n    """uint8 (N, 3, H, W) -> normalised float tensor."""\n    x = slices.float().div_(255.0) if slices.dtype == torch.uint8 else slices.float()\n    mean = torch.tensor(MEAN, device=x.device).view(1, 3, 1, 1)\n    std = torch.tensor(STD, device=x.device).view(1, 3, 1, 1)\n    return (x - mean) / std\n\n\ndef pool_patches(patches: "torch.Tensor", grid: int) -> "torch.Tensor":\n    """(N, P, C) patch tokens -> (N, grid*grid*C), spatially pooled.\n\n    The patch sequence is reshaped back to its square grid before pooling, so\n    neighbouring patches are averaged with their neighbours rather than with\n    whatever happened to sit beside them in a flattened sequence.\n    """\n    import torch.nn.functional as F\n\n    count, n_patches, channels = patches.shape\n    side = int(round(n_patches ** 0.5))\n    if side * side != n_patches:\n        # Not a square grid; fall back to the mean rather than mis-shaping it.\n        return patches.mean(dim=1)\n\n    square = patches.transpose(1, 2).reshape(count, channels, side, side)\n    pooled = F.adaptive_avg_pool2d(square, grid)\n    return pooled.flatten(2).transpose(1, 2).reshape(count, grid * grid * channels)\n\n\ndef embed_slices(encoder, slices, grid: int = None) -> "torch.Tensor":\n    """(N, 3, H, W) -> (N, embed_dim(grid)): CLS plus pooled patch tokens.\n\n    Uses forward_features so the patch tokens are reachable at all; the model\'s\n    default pooled output discards them.\n    """\n    grid = POOL_GRID if grid is None else grid\n\n    with torch.no_grad():\n        tokens = encoder.forward_features(normalize(slices))\n\n        if tokens.ndim == 2:\n            # A backbone that already pools; repeat so the width is predictable.\n            repeats = embed_dim(grid) // tokens.shape[-1]\n            return tokens.repeat(1, repeats)\n\n        cls = tokens[:, 0]\n        prefix = getattr(encoder, "num_prefix_tokens", 1)\n        patches = tokens[:, prefix:]\n\n        if grid > 1:\n            return torch.cat([cls, pool_patches(patches, grid)], dim=-1)\n        return torch.cat([cls, patches.mean(dim=1)], dim=-1)\n\n\nclass CachedEmbeddingBackbone(nn.Module if nn is not None else object):\n    """Adapter so KneeNet can consume precomputed embeddings unchanged.\n\n    KneeNet expects a module mapping slices to features. When features are\n    already computed, the "backbone" is an identity that reshapes -- which\n    keeps one model definition for both the cached and the end-to-end path.\n    """\n\n    def __init__(self, feature_dim: int = EMBED_DIM):\n        super().__init__()\n        self.feature_dim = feature_dim\n\n    def forward(self, x):\n        return x\n',
    'src/model/dataset.py': '"""Study-level dataset over the cached volumes.\n\nJoins four sources into one training sample per study:\n\n    cache shards      the uint8 volumes, memory-mapped\n    manifest          which shard row holds which series\n    fused labels      12 soft targets and 12 sample weights per study\n    side predictions  L/R per series, from the tag or the classifier\n\nA sample is a whole study, not a series. The twelve findings are study-level\nlabels, and a single series often cannot support them -- effusion reads off a\nfluid-sensitive sequence, patellofemoral OA off a different one -- so the model\nhas to see the study\'s series together.\n\n**Laterality is handled two ways, depending on the path.**\n\nFor the end-to-end path, `mirror_volume` reflects left knees onto the canonical\nside at load time, along the axis the acquisition plane dictates: slice order\nfor sagittal, where medial-lateral is the slice axis, and columns for coronal\nand axial, where it lies in-plane. Doing it at load time rather than in the\ncache means side detection can improve without rebuilding 29 GB.\n\nFor the frozen path this is not available -- the embeddings are computed from\nthe volume, so the image can no longer be flipped. There the side travels to\nthe model as a covariate (`is_right`, `side_known`) instead. That still\naddresses the original concern, which was that the model should not have to\n*infer* the side from cues that are not reliably present; it is handed the\nanswer either way. Which of the two performs better is an experiment, not a\nsettled question.\n"""\n\nfrom typing import Dict, List, Optional, Sequence\n\nimport numpy as np\nimport pandas as pd\n\nfrom src import config\n\n# Plane -> which axis of a (slices, rows, cols) volume runs medial-to-lateral.\nPLANE_MIRROR_AXIS = {\n    "Sagittal": 0,   # medial-lateral is the slice axis\n    "Coronal": 2,    # medial-lateral lies in-plane\n    "Axial": 2,\n}\nDEFAULT_MIRROR_AXIS = 2\n\n# Series metadata fed to the model as conditioning. Findings are\n# sequence-specific, and these three are given in both train and test.\n# `is_right` and `side_known` carry laterality as a covariate rather than\n# normalising it away by mirroring. With precomputed frozen embeddings the\n# image can no longer be flipped -- the features are already computed -- so the\n# side is handed to the model instead, which addresses the original concern\n# (the model should not have to *infer* the side) by a different route.\n# Mirroring remains available for the end-to-end path and is an A/B to run,\n# not a settled question.\nMETA_COLUMNS = ["Fluid_Sensitive", "Fat_Suppression", "is_sagittal",\n                "is_coronal", "is_axial", "is_right", "side_known"]\n\n\ndef mirror_volume(volume: np.ndarray, side: Optional[str],\n                  plane: Optional[str]) -> np.ndarray:\n    """Reflect a left knee onto the canonical side, along the correct axis."""\n    if side is None or (isinstance(side, float) and np.isnan(side)):\n        # Unknown side: leave as acquired. Guessing swaps medial with lateral\n        # on four of the twelve findings.\n        return volume\n    if str(side).upper() == config.CANONICAL_SIDE:\n        return volume\n\n    axis = PLANE_MIRROR_AXIS.get(str(plane), DEFAULT_MIRROR_AXIS)\n    return np.flip(volume, axis=axis).copy()\n\n\ndef series_metadata(row: pd.Series) -> np.ndarray:\n    """Conditioning vector for one series.\n\n    Fluid sensitivity and fat suppression are kept separate: the data\n    description notes they are often correlated but not equivalent, so\n    collapsing them would discard the cases where they disagree.\n    """\n    plane = str(row.get("Anatomical_Plane", ""))\n    side = row.get("side_predicted")\n    known = isinstance(side, str) and side.upper() in ("L", "R")\n    return np.array([\n        float(row.get("Fluid_Sensitive", 0) or 0),\n        float(row.get("Fat_Suppression", 0) or 0),\n        float(plane == "Sagittal"),\n        float(plane == "Coronal"),\n        float(plane == "Axial"),\n        float(known and side.upper() == "R"),\n        float(known),\n    ], dtype=np.float32)\n\n\nclass StudyIndex:\n    """Everything needed to assemble a study, resolved once up front.\n\n    Built from frames rather than files so it can be constructed in a test\n    without touching disk.\n    """\n\n    def __init__(self, manifest: pd.DataFrame, series_meta: pd.DataFrame,\n                 labels: Optional[pd.DataFrame] = None,\n                 sides: Optional[pd.DataFrame] = None,\n                 max_series: Optional[int] = None,\n                 reports: Optional[pd.DataFrame] = None):\n        frame = manifest.merge(\n            series_meta[["SeriesInstanceUID", "Fluid_Sensitive",\n                         "Fat_Suppression", "Anatomical_Plane"]],\n            on="SeriesInstanceUID", how="left")\n\n        if sides is not None:\n            frame = frame.merge(\n                sides[["SeriesInstanceUID", "side_predicted"]],\n                on="SeriesInstanceUID", how="left")\n        else:\n            frame["side_predicted"] = frame.get("laterality_resolved")\n\n        # Drop series whose volume failed to build; their row in the cache is\n        # zeros and would train the model on blank images.\n        if "error" in frame.columns:\n            frame = frame[frame["error"].isna()].reset_index(drop=True)\n\n        self.series = frame\n        self.max_series = max_series\n        self.labels = labels\n\n        self.studies: List[str] = sorted(frame[config.ID_COL].unique())\n        self._by_study = {\n            study: group.reset_index(drop=True)\n            for study, group in frame.groupby(config.ID_COL)\n        }\n\n        if labels is not None:\n            keyed = labels.set_index(config.ID_COL)\n            self.studies = [s for s in self.studies if s in keyed.index]\n            self._labels = keyed\n        else:\n            self._labels = None\n\n        # Report sentence embeddings, a training-time-only signal: reports\n        # exist for train studies and never for test, so these feed an\n        # auxiliary loss rather than the model input. Stored as one float32\n        # matrix rather than a DataFrame so __getitem__ is an array row.\n        if reports is not None:\n            keyed_reports = reports.set_index(config.ID_COL)\n            self._report_rows = {study: i for i, study\n                                 in enumerate(keyed_reports.index)}\n            self._report_matrix = keyed_reports.to_numpy(dtype=np.float32)\n            self.report_dim = self._report_matrix.shape[1]\n        else:\n            self._report_rows = None\n            self._report_matrix = None\n            self.report_dim = None\n\n    def __len__(self) -> int:\n        return len(self.studies)\n\n    def series_for(self, study: str) -> pd.DataFrame:\n        """Series of a study, capped deterministically when a cap is set.\n\n        The cap exists for the efficiency track: inference cost scales with\n        series count, and a study with a dozen series contributes far more\n        runtime than one with three. Selection is by position rather than at\n        random so a prediction is reproducible.\n        """\n        rows = self._by_study[study]\n        if self.max_series is not None and len(rows) > self.max_series:\n            keep = np.linspace(0, len(rows) - 1, self.max_series).round().astype(int)\n            rows = rows.iloc[np.unique(keep)].reset_index(drop=True)\n        return rows\n\n    def targets_for(self, study: str):\n        """(targets, weights) for a study, or (None, None) when unlabelled."""\n        if self._labels is None or study not in self._labels.index:\n            return None, None\n        row = self._labels.loc[study]\n        targets = np.array([float(row[name]) for name in config.LABELS],\n                           dtype=np.float32)\n        weights = np.array(\n            [float(row.get(f"{name}_weight", 1.0)) for name in config.LABELS],\n            dtype=np.float32)\n        return targets, weights\n\n    def report_for(self, study: str) -> Optional[np.ndarray]:\n        """The study\'s report embedding, or None when unavailable."""\n        if self._report_rows is None or study not in self._report_rows:\n            return None\n        return self._report_matrix[self._report_rows[study]]\n\n\ndef load_study(index: StudyIndex, study: str, shards: Dict[str, np.ndarray]) -> Dict:\n    """Assemble one study: mirrored volumes, conditioning, targets, weights."""\n    rows = index.series_for(study)\n\n    volumes, meta = [], []\n    for _, row in rows.iterrows():\n        shard = shards[row["cache_path"]]\n        volume = np.asarray(shard[int(row["position"])])\n        volume = mirror_volume(volume, row.get("side_predicted"),\n                               row.get("Anatomical_Plane"))\n        volumes.append(volume)\n        meta.append(series_metadata(row))\n\n    targets, weights = index.targets_for(study)\n    return {\n        config.ID_COL: study,\n        "volumes": np.stack(volumes) if volumes else np.zeros(\n            (0, config.CACHE_SLICES, config.CACHE_SIZE, config.CACHE_SIZE),\n            dtype=np.uint8),\n        "meta": np.stack(meta) if meta else np.zeros((0, len(META_COLUMNS)),\n                                                     dtype=np.float32),\n        "targets": targets,\n        "weights": weights,\n        "report": index.report_for(study),\n    }\n\n\ndef load_study_features(index: StudyIndex, study: str,\n                        shards: Dict[str, np.ndarray]) -> Dict:\n    """Assemble one study from precomputed slice embeddings.\n\n    Mirroring is unavailable here -- the features were computed from the image\n    and cannot be flipped after the fact -- so laterality reaches the model\n    through `series_metadata` instead.\n    """\n    rows = index.series_for(study)\n\n    features, meta = [], []\n    for _, row in rows.iterrows():\n        shard = shards[row["cache_path"]]\n        features.append(np.asarray(shard[int(row["position"])], dtype=np.float32))\n        meta.append(series_metadata(row))\n\n    targets, weights = index.targets_for(study)\n    return {\n        config.ID_COL: study,\n        "volumes": np.stack(features) if features else np.zeros((0, 1, 1),\n                                                                dtype=np.float32),\n        "meta": np.stack(meta) if meta else np.zeros((0, len(META_COLUMNS)),\n                                                     dtype=np.float32),\n        "targets": targets,\n        "weights": weights,\n        "report": index.report_for(study),\n    }\n\n\ndef collate_studies(batch: Sequence[Dict]) -> Dict:\n    """Pad a batch of studies to the largest series count, with a mask.\n\n    Series counts vary from one to a dozen, so the batch is padded and masked\n    rather than truncated. The mask must reach the attention pooling: without\n    it, padding rows are averaged in as if they were real series.\n    """\n    import torch\n\n    longest = max(len(item["volumes"]) for item in batch)\n\n    # Per-series shape is taken from the data rather than assumed, so the same\n    # collate serves raw volumes (D, H, W) and cached embeddings (D, F).\n    reference = next((item["volumes"] for item in batch if len(item["volumes"])),\n                     None)\n    if reference is None:\n        raise ValueError("every study in the batch is empty")\n    per_series = reference.shape[1:]\n    dtype = reference.dtype\n\n    volumes = np.zeros((len(batch), longest, *per_series), dtype=dtype)\n    meta = np.zeros((len(batch), longest, len(META_COLUMNS)), dtype=np.float32)\n    mask = np.zeros((len(batch), longest), dtype=bool)\n\n    for i, item in enumerate(batch):\n        count = len(item["volumes"])\n        if count:\n            volumes[i, :count] = item["volumes"]\n            meta[i, :count] = item["meta"]\n            mask[i, :count] = True\n\n    out = {\n        config.ID_COL: [item[config.ID_COL] for item in batch],\n        "volumes": torch.from_numpy(volumes),\n        "meta": torch.from_numpy(meta),\n        "mask": torch.from_numpy(mask),\n    }\n\n    if batch[0]["targets"] is not None:\n        out["targets"] = torch.from_numpy(\n            np.stack([item["targets"] for item in batch]))\n        out["weights"] = torch.from_numpy(\n            np.stack([item["weights"] for item in batch]))\n\n    # Report embeddings, present only where the index carries them. Studies\n    # without one get a zero row and a False mask so the auxiliary loss can\n    # exclude them without a ragged batch.\n    reports = [item.get("report") for item in batch]\n    dims = [r.shape[0] for r in reports if r is not None]\n    if dims:\n        report = np.zeros((len(batch), dims[0]), dtype=np.float32)\n        report_mask = np.zeros(len(batch), dtype=bool)\n        for i, row in enumerate(reports):\n            if row is not None:\n                report[i] = row\n                report_mask[i] = True\n        out["report"] = torch.from_numpy(report)\n        out["report_mask"] = torch.from_numpy(report_mask)\n    return out\n',
    'src/model/net.py': '"""Study-level model: slices to series to study to twelve logits.\n\nThe backbone is deliberately the least interesting part. It is swappable, and\nwhich one wins is settled by measurement rather than by argument -- at 4,349\nstudies with mined labels, the aggregation above it matters more.\n\n    (B, S, D, H, W) uint8 volumes\n      -> backbone over slices          (B*S*D, F)\n      -> attention pool over slices    (B*S, F)\n      -> FiLM on sequence metadata     (B*S, F)\n      -> transformer over series       (B, F)      masked\n      -> head                          (B, 12)\n\n**Why FiLM on the sequence metadata.** Findings are sequence-specific: effusion\nand Baker\'s cysts read off fluid-sensitive series, cartilage loss off\nfat-suppressed ones, ACL tears off sagittals. `Fluid_Sensitive`,\n`Fat_Suppression`, and the plane are given in both train and test, so pooling\nevery series through one undifferentiated path throws away free signal.\n\n**Why masking is load-bearing.** Series counts run from one to a dozen. Padded\nrows must be excluded from both the attention pool and the transformer, or a\nshort study is diluted by rows of zeros that the model reads as real series.\n"""\n\nfrom typing import Optional, Sequence\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom src import config\nfrom src.model.dataset import META_COLUMNS\n\n# Adjacent slices stacked into the three channels an ImageNet-pretrained\n# backbone expects. Gives each slice a little through-plane context for free.\nSLICE_CONTEXT = 3\n\n\ndef stack_2p5d(volumes: torch.Tensor) -> torch.Tensor:\n    """(N, D, H, W) -> (N*D, 3, H, W) using each slice and its neighbours.\n\n    Edge slices repeat, which is preferable to zero padding: a black plane at\n    the edge of a volume is a feature the model would learn to key on.\n    """\n    n, depth, height, width = volumes.shape\n    previous = torch.cat([volumes[:, :1], volumes[:, :-1]], dim=1)\n    following = torch.cat([volumes[:, 1:], volumes[:, -1:]], dim=1)\n    stacked = torch.stack([previous, volumes, following], dim=2)\n    return stacked.reshape(n * depth, SLICE_CONTEXT, height, width)\n\n\n# The statistics DINOv2 was pretrained under. `embed.py` has always applied\n# these on the frozen path; the end-to-end path fed raw [0,1] and ConvNeXt\n# absorbed the shift into its first layer over twelve epochs. A pre-norm ViT\n# is far less forgiving, so the same trunk was being fed a distribution it\n# never saw during pretraining.\nIMAGENET_MEAN = (0.485, 0.456, 0.406)\nIMAGENET_STD = (0.229, 0.224, 0.225)\n\n# NORMALISATION_BUFFERS -- why input_mean/input_std are persistent.\n#\n# They were registered with persistent=False on the argument that they are\n# constants of the backbone family rather than learned state. That was true\n# and it was still wrong: absent from the state dict, they have to be supplied\n# again at every construction site, a site that forgets raises nothing, and a\n# strict load restores every weight while leaving normalisation off. Two\n# scoring sites forgot. The 24-slice DINOv2 run logged gold 0.864 from the\n# in-memory model and 0.823 from the same weights reloaded -- an\n# ImageNet-normalised trunk scored on raw [0,1].\n#\n# So they travel now, and `KneeNet._load_from_state_dict` handles both\n# directions: a checkpoint that carries them imposes them on whatever the\n# caller built, and one written before this change loads strictly without\n# them. ConvNeXt is unaffected either way -- its defaults are None, so its\n# state dict gains no keys and every existing reference number stands.\n\n# How far below the head\'s learning rate a pretrained trunk should sit. The\n# optimiser puts every parameter in one group at OneCycle\'s peak of\n# `lr * max_lr_multiplier` = 3e-4, which a supervised ConvNeXt survives and a\n# self-supervised ViT does not: it loses its pretrained features during warmup\n# and then relearns them from 4,349 studies, which is the one regime ViTs are\n# worst in. Measured symptom: gold 0.6 at epoch 4 where ConvNeXt reached\n# 0.729, and *below* the same trunk frozen (0.770).\nVIT_BACKBONE_LR_SCALE = 0.1\n\n\ndef backbone_defaults(name: str) -> dict:\n    """Input normalisation and trunk learning-rate scale for a backbone family.\n\n    Returns ConvNeXt\'s existing behaviour unchanged -- no normalisation, one\n    learning rate -- so the 0.861 fold-1 reference stays comparable. That\n    includes ConvNeXt-V2, whose FCMAE weights are a drop-in for V1 at the same\n    `timm` interface, and DINOv3\'s ConvNeXt variants: keeping the convnet path\n    identical is what makes a backbone swap a one-variable experiment.\n\n    ViT-family trunks are the exception on both axes, and both were measured\n    rather than argued -- see VIT_BACKBONE_LR_SCALE.\n    """\n    lowered = name.lower()\n    is_vit = "vit" in lowered or ("dinov" in lowered and "convnext" not in lowered)\n    if is_vit:\n        return {"input_mean": IMAGENET_MEAN, "input_std": IMAGENET_STD,\n                "backbone_lr_scale": VIT_BACKBONE_LR_SCALE}\n    return {"input_mean": None, "input_std": None, "backbone_lr_scale": 1.0}\n\n\ndef build_backbone(name: str, pretrained: bool = True,\n                   image_size: Optional[int] = None):\n    """Create a timm trunk mapping (N, 3, H, W) -> (N, num_features).\n\n    Every backbone this project runs is constructed through here so the\n    training and submission notebooks cannot build different trunks from the\n    same name -- notebook 10 hard-coded ConvNeXt for a while, which would have\n    met a DINOv2 checkpoint with a shape error at submission time.\n\n    ViT-family weights (DINOv2 among them) are published at a fixed input size\n    and timm\'s patch embedding refuses anything else, so the cache\'s size is\n    passed explicitly and the position embeddings are interpolated -- 224 is\n    16x14, which divides cleanly. ConvNets accept any size and reject the\n    argument, hence the fallback rather than an unconditional pass.\n    """\n    import timm\n\n    image_size = config.CACHE_SIZE if image_size is None else image_size\n    kwargs = dict(pretrained=pretrained, num_classes=0,\n                  in_chans=SLICE_CONTEXT)\n    try:\n        return timm.create_model(name, img_size=image_size, **kwargs)\n    except TypeError:\n        return timm.create_model(name, **kwargs)\n\n\nclass AttentionPool(nn.Module):\n    """Masked attention pooling over a sequence of embeddings.\n\n    Preferred to mean pooling because the informative slices are a minority --\n    a meniscal tear appears on a few slices of a stack of 24, and averaging\n    dilutes it by the rest. This is the attention-based MIL pooling of Ilse et\n    al., "Attention-based Deep Multiple Instance Learning" (ICML 2018,\n    arXiv:1802.04712), equation 8: a trainable weighted average, permutation\n    invariant, with the weights produced by a two-layer scorer.\n\n    `gated=True` selects the same paper\'s equation 9, which replaces the single\n    tanh branch with `tanh(Vh) * sigmoid(Uh)`. The stated motivation is that\n    tanh is close to linear over most of its range, so it cannot express the\n    non-linear "how relevant is this instance" function well; the sigmoid\n    branch supplies a multiplicative gate that can suppress an instance\n    outright. It is a strictly larger family -- a gate saturated at 1\n    reproduces the ungated pool -- and it costs one extra `dim x hidden`\n    matrix per pool.\n\n    The two variants produce different state dict keys on purpose. A gated\n    checkpoint loaded into an ungated model raises on the missing keys rather\n    than scoring quietly through the wrong pooling, which is the failure mode\n    every other geometry mismatch in this project has had.\n    """\n\n    def __init__(self, dim: int, hidden: int = 128, gated: bool = False):\n        super().__init__()\n        self.gated = gated\n        if gated:\n            self.attend = nn.Linear(dim, hidden)\n            self.gate = nn.Linear(dim, hidden)\n            self.weigh = nn.Linear(hidden, 1)\n        else:\n            self.score = nn.Sequential(\n                nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))\n\n    def scores(self, x: torch.Tensor) -> torch.Tensor:\n        """(B, N, F) -> (B, N) unnormalised attention logits."""\n        if not self.gated:\n            return self.score(x).squeeze(-1)\n        return self.weigh(torch.tanh(self.attend(x))\n                          * torch.sigmoid(self.gate(x))).squeeze(-1)\n\n    def forward(self, x: torch.Tensor,\n                mask: Optional[torch.Tensor] = None) -> torch.Tensor:\n        # x: (B, N, F), mask: (B, N) with True for real entries.\n        weights = self.scores(x)\n        if mask is not None:\n            weights = weights.masked_fill(~mask, float("-inf"))\n        weights = torch.softmax(weights, dim=1).unsqueeze(-1)\n        return (x * weights).sum(dim=1)\n\n\nclass FiLM(nn.Module):\n    """Feature-wise modulation of series embeddings by sequence metadata."""\n\n    def __init__(self, dim: int, meta_dim: int = len(META_COLUMNS)):\n        super().__init__()\n        self.to_scale_shift = nn.Sequential(\n            nn.Linear(meta_dim, dim), nn.ReLU(inplace=True),\n            nn.Linear(dim, dim * 2))\n\n    def forward(self, x: torch.Tensor, meta: torch.Tensor) -> torch.Tensor:\n        scale, shift = self.to_scale_shift(meta).chunk(2, dim=-1)\n        # Centre the scale on 1 so an untrained FiLM is close to the identity.\n        return x * (1.0 + scale) + shift\n\n\nclass KneeNet(nn.Module):\n    """Backbone plus the aggregation stack, emitting twelve study-level logits.\n\n    `backbone` is any module mapping (N, 3, H, W) to (N, feature_dim). Passing\n    a frozen encoder is supported and is the intended first configuration: with\n    the trunk frozen, slice embeddings are computed once and the stack above\n    trains in minutes rather than hours.\n    """\n\n    def __init__(self, backbone: Optional[nn.Module], feature_dim: int,\n                 n_labels: int = None, series_layers: int = 2,\n                 n_heads: int = 8, dropout: float = 0.2,\n                 freeze_backbone: bool = False,\n                 input_dim: Optional[int] = None,\n                 input_mean=None, input_std=None,\n                 gated_pool: bool = False,\n                 residual_pairs=None, residual_beta=None):\n        super().__init__()\n        # None is valid on the frozen path, where the trunk ran offline.\n        self.backbone = backbone if backbone is not None else nn.Identity()\n        self.feature_dim = feature_dim\n        n_labels = n_labels or len(config.LABELS)\n\n        if freeze_backbone:\n            for parameter in self.backbone.parameters():\n                parameter.requires_grad = False\n\n        # Spatially pooled embeddings are wide (6,528 for a 4x4 grid), and\n        # running the whole stack at that width would put ~85M parameters in\n        # FiLM alone. A projection keeps the aggregation at a sane size.\n        self.project = (nn.Linear(input_dim, feature_dim)\n                        if input_dim and input_dim != feature_dim\n                        else nn.Identity())\n\n        # These now travel in the state dict, and `_load_from_state_dict`\n        # below reconciles the two directions with older files. See\n        # NORMALISATION_BUFFERS for why the earlier persistent=False cost a\n        # measured run.\n        #\n        # Registered as None rather than assigned as a plain attribute: a\n        # plain attribute of the same name makes a later register_buffer\n        # raise "attribute already exists", and adopting a checkpoint\'s\n        # statistics is exactly that late registration.\n        if input_mean is not None:\n            self.register_buffer(\n                "input_mean", torch.tensor(input_mean).view(1, -1, 1, 1))\n            self.register_buffer(\n                "input_std", torch.tensor(input_std).view(1, -1, 1, 1))\n        else:\n            self.register_buffer("input_mean", None)\n            self.register_buffer("input_std", None)\n\n        self.slice_pool = AttentionPool(feature_dim, gated=gated_pool)\n        self.film = FiLM(feature_dim)\n\n        layer = nn.TransformerEncoderLayer(\n            d_model=feature_dim, nhead=n_heads,\n            dim_feedforward=feature_dim * 2, dropout=dropout,\n            batch_first=True, norm_first=True)\n        self.series_encoder = nn.TransformerEncoder(layer, num_layers=series_layers)\n        self.study_pool = AttentionPool(feature_dim, gated=gated_pool)\n\n        self.head = nn.Sequential(\n            nn.LayerNorm(feature_dim),\n            nn.Dropout(dropout),\n            nn.Linear(feature_dim, n_labels))\n\n        # Residual parameterisation: for each (child, parent) pair,\n        #\n        #     child_logit = beta * stopgrad(parent_logit) + g(features)\n        #\n        # with g the head\'s own child column. The plan\'s Stage 1: the\n        # synovitis head reconstructs from the effusion head at R^2 0.55 with\n        # a coefficient of +0.843, so effusion is most of what it computes.\n        # Handing it that term for free, detached, leaves g nothing to learn\n        # except the part effusion does not explain -- which is the part the\n        # conditional metric measures.\n        #\n        # `beta` is calibrated on out-of-fold predictions, not derived from\n        # r_true = 0.403: a rank correlation between two binary labels does\n        # not map onto a logit coefficient. See\n        # `src.model.conditional.calibrate_coupling`.\n        #\n        # Registered as None when unconfigured, exactly like the\n        # normalisation buffers above and for the same reason -- a run that\n        # does not use this writes an unchanged state dict, and a run that\n        # does carries the coupling into the file so a bare\n        # `KneeNet(backbone, dim)` at a scoring site reproduces it instead of\n        # silently scoring an uncoupled model.\n        if residual_pairs:\n            pairs = torch.tensor(list(residual_pairs), dtype=torch.long)\n            if pairs.ndim != 2 or pairs.shape[1] != 2:\n                raise ValueError(\n                    "residual_pairs must be (child, parent) index pairs, got "\n                    f"shape {tuple(pairs.shape)}")\n            if residual_beta is None:\n                raise ValueError("residual_pairs given without residual_beta")\n            values = (list(residual_beta) if hasattr(residual_beta, "__len__")\n                      else [float(residual_beta)] * len(pairs))\n            beta = torch.tensor(values, dtype=torch.float32)\n            if len(beta) != len(pairs):\n                raise ValueError(\n                    f"residual_beta has {len(beta)} values for "\n                    f"{len(pairs)} pairs")\n            self.register_buffer("residual_pairs", pairs)\n            self.register_buffer("residual_beta", beta)\n        else:\n            self.register_buffer("residual_pairs", None)\n            self.register_buffer("residual_beta", None)\n\n    def _load_from_state_dict(self, state_dict, prefix, local_metadata, strict,\n                              missing_keys, unexpected_keys, error_msgs):\n        """Reconcile the normalisation buffers with checkpoints either side.\n\n        Three cases, and the middle one is the bug this exists for:\n\n        - **The file carries the statistics and this model has none.** Adopt\n          them. A bare `KneeNet(backbone, dim)` built at a scoring site now\n          normalises the way the run that wrote the file did, instead of\n          feeding an ImageNet-normalised trunk raw [0,1] and reading as a\n          worse checkpoint.\n        - **The file predates this change and this model has statistics.**\n          Keep what the caller passed. Every checkpoint written before\n          2026-08-28 is missing the two keys, and a strict load must not\n          start failing on files this project already shipped.\n        - **The file carries statistics and this model has different ones.**\n          The file wins, for the same reason as the first case: the weights\n          and the statistics they were trained under belong together, and the\n          construction site is the thing that has been wrong before.\n\n        Adopting is a `register_buffer` on a slot registered as None in\n        `__init__`, which is why it is registered there rather than assigned.\n        """\n        for name in ("input_mean", "input_std",\n                     "residual_pairs", "residual_beta"):\n            key = prefix + name\n            if key in state_dict:\n                self.register_buffer(name, state_dict[key].clone())\n            elif self._buffers.get(name) is not None:\n                # Pre-fill so the strict load below sees no missing key.\n                state_dict[key] = self._buffers[name]\n        super()._load_from_state_dict(\n            state_dict, prefix, local_metadata, strict, missing_keys,\n            unexpected_keys, error_msgs)\n\n    def encode_series(self, volumes: torch.Tensor) -> torch.Tensor:\n        """(N, D, H, W) uint8 or float -> (N, F) one embedding per series."""\n        count, depth = volumes.shape[0], volumes.shape[1]\n        if volumes.dtype == torch.uint8:\n            volumes = volumes.float().div_(255.0)\n\n        slices = stack_2p5d(volumes)\n        if self.input_mean is not None:\n            slices = (slices - self.input_mean) / self.input_std\n        features = self.backbone(slices)\n        features = features.reshape(count, depth, self.feature_dim)\n        return self.slice_pool(features)\n\n    def pool_study(self, series: torch.Tensor, meta: torch.Tensor,\n                   mask: torch.Tensor) -> torch.Tensor:\n        """(B, S, F) series embeddings -> (B, F) one embedding per study.\n\n        The study representation the head reads. Exposed separately so\n        auxiliary objectives (report-embedding regression) can supervise the\n        representation without their parameters entering this module -- the\n        exported state dict must stay loadable by a bare `KneeNet`.\n        """\n        series = self.film(series, meta)\n\n        # The transformer takes True for positions to IGNORE, the inverse of\n        # our mask.\n        encoded = self.series_encoder(series, src_key_padding_mask=~mask)\n        # A fully padded row emits NaN from the attention; zero it before\n        # pooling so one short study cannot poison the batch.\n        encoded = torch.nan_to_num(encoded)\n\n        return self.study_pool(encoded, mask=mask)\n\n    def couple(self, logits: torch.Tensor) -> torch.Tensor:\n        """Add the detached parent term to each configured child logit.\n\n        A no-op unless the model was built (or loaded) with residual pairs, so\n        every checkpoint written before this existed behaves identically.\n\n        The parent term is detached: the point is to relieve the child of\n        learning effusion, not to let the synovitis objective reach back and\n        reshape the effusion head, which is the one head on this finding that\n        is reliable.\n        """\n        if self.residual_pairs is None:\n            return logits\n        child = self.residual_pairs[:, 0]\n        parent = self.residual_pairs[:, 1]\n        addition = torch.zeros_like(logits)\n        addition[:, child] = (self.residual_beta.to(logits.dtype)\n                              * logits[:, parent].detach())\n        return logits + addition\n\n    def aggregate(self, series: torch.Tensor, meta: torch.Tensor,\n                  mask: torch.Tensor) -> torch.Tensor:\n        """(B, S, F) series embeddings -> (B, n_labels).\n\n        Everything above the backbone. Shared by the end-to-end path and the\n        frozen path so the two cannot drift into different architectures.\n        """\n        return self.couple(self.head(self.pool_study(series, meta, mask)))\n\n    def forward_features(self, features: torch.Tensor, meta: torch.Tensor,\n                         mask: torch.Tensor) -> torch.Tensor:\n        """Frozen path: (B, S, D, F) precomputed slice embeddings.\n\n        The backbone has already run, so this starts at the slice pooling.\n        """\n        batch, series, depth, _ = features.shape\n        projected = self.project(features.reshape(batch * series, depth, -1))\n        pooled = self.slice_pool(projected)\n        return self.aggregate(pooled.reshape(batch, series, self.feature_dim),\n                              meta, mask)\n\n    def forward_pooled(self, volumes: torch.Tensor, meta: torch.Tensor,\n                       mask: torch.Tensor):\n        """End-to-end path returning (logits, pooled study embedding).\n\n        The training loop wants both: logits for the label losses, the pooled\n        representation for auxiliary supervision. Computing them together\n        avoids a second pass through the backbone.\n        """\n        batch, series = volumes.shape[0], volumes.shape[1]\n        flat = volumes.reshape(batch * series, *volumes.shape[2:])\n        embeddings = self.encode_series(flat).reshape(batch, series, self.feature_dim)\n        pooled = self.pool_study(embeddings, meta, mask)\n        return self.couple(self.head(pooled)), pooled\n\n    def forward(self, volumes: torch.Tensor, meta: torch.Tensor,\n                mask: torch.Tensor) -> torch.Tensor:\n        # volumes: (B, S, D, H, W), meta: (B, S, M), mask: (B, S)\n        logits, _ = self.forward_pooled(volumes, meta, mask)\n        return logits\n\n\ndef weighted_bce(logits: torch.Tensor, targets: torch.Tensor,\n                 weights: torch.Tensor, label_smoothing: float = 0.0,\n                 pos_weight: Optional[torch.Tensor] = None,\n                 symmetric_beta: float = 0.0) -> torch.Tensor:\n    """BCE against soft targets, weighted per study and finding.\n\n    Targets are probabilities rather than 0/1 -- they come from two extractors\n    that disagree often enough that thresholding would teach the model to\n    reproduce extraction errors. Weights carry how much each target is\n    believed: full for agreement, less for conflict, least for silence.\n\n    Three optional terms, all inert at their defaults so the reference runs\n    stay bit-comparable. Each addresses something the plan names:\n\n    `label_smoothing` pulls every target toward 0.5 by `eps`. The mined labels\n    disagree with image-derived ground truth about 18% of the time, and a\n    confident target that is wrong 18% of the time is what late training\n    memorises. Note that soft targets already carry some of this effect --\n    smoothing on top of them is a floor on the target\'s distance from 0.5, not\n    the usual one-hot correction.\n\n    `pos_weight` scales the positive term per finding. The metric is macro\n    AUC, so a rare finding costs exactly as much as a common one; without this\n    the loss is dominated by whatever is prevalent. Passed straight through to\n    torch, which applies it to soft targets correctly.\n\n    `symmetric_beta` adds the reverse cross-entropy term of Wang et al.,\n    "Symmetric Cross Entropy for Robust Learning with Noisy Labels" (ICCV\n    2019). RCE swaps the roles of prediction and target, which bounds the loss\n    a wrong label can contribute: a confidently mislabelled study cannot drag\n    the gradient the way plain CE lets it. Targets are clamped off 0 and 1\n    because RCE takes their log; the clamp is what the paper\'s `A` constant\n    does.\n    """\n    if label_smoothing > 0:\n        targets = targets * (1.0 - label_smoothing) + 0.5 * label_smoothing\n\n    loss = F.binary_cross_entropy_with_logits(\n        logits, targets, pos_weight=pos_weight, reduction="none")\n\n    if symmetric_beta > 0:\n        probs = torch.sigmoid(logits)\n        clamped = targets.clamp(1e-4, 1.0 - 1e-4)\n        reverse = -(probs * torch.log(clamped)\n                    + (1.0 - probs) * torch.log1p(-clamped))\n        loss = loss + symmetric_beta * reverse\n\n    total = weights.sum()\n    if total <= 0:\n        return loss.mean() * 0.0\n    return (loss * weights).sum() / total\n\n\ndef pairwise_rank_loss(scores: torch.Tensor, targets: torch.Tensor,\n                       ref_scores: torch.Tensor, ref_targets: torch.Tensor,\n                       weights: Optional[torch.Tensor] = None,\n                       ref_weights: Optional[torch.Tensor] = None,\n                       min_gap: float = 0.25,\n                       pair_mask: Optional[torch.Tensor] = None,\n                       max_margin: Optional[float] = None) -> torch.Tensor:\n    """Per-finding RankNet loss between a batch and a reference set.\n\n    The metric is macro AUC -- a ranking statistic -- while BCE optimises\n    calibration; this term optimises the ordering directly. Pairs are formed\n    per finding between the current batch (`scores`/`targets`, (B, L), with\n    gradient) and a reference set ((Q, L), typically detached scores from\n    recent steps), because at one study per rank there are no pairs *within*\n    a batch to rank.\n\n    Only pairs whose soft targets differ by at least `min_gap` count: the\n    targets are extractor probabilities, and ordering two studies the\n    extractor itself barely distinguishes would train the model to reproduce\n    extraction noise. Surviving pairs are weighted by the target gap and by\n    both studies\' label weights, so a confident 0.9-vs-0.1 pair outweighs a\n    marginal one. Returns a zero that still touches `scores` when no pair\n    qualifies, so the graph is identical either way.\n\n    `pair_mask` drops pairs before weighting. Broadcastable to (B, Q, L), so\n    it can restrict which findings are ranked, which pairs are comparable, or\n    both. `stratified_rank_loss` below builds one; at None every pair counts\n    and this is the loss the reference runs used, unchanged.\n\n    `max_margin` clamps the logit difference before the softplus, and exists\n    because leaving it unbounded diverged a real run (2026-08-31). A satisfied\n    pair\'s loss decays to zero as its margin grows, but a **violated** pair\'s\n    grows linearly and forever, so when the pair constraints contradict each\n    other -- which is what within-stratum synovitis pairs do, since 90% of the\n    mined targets sit at zero and the surviving pairs are mostly "mentioned vs\n    silent" -- inflating the logit scale reduces loss on the majority faster\n    than it raises it on the minority. The head then saturates, every study\n    ties, AUC lands on exactly 0.5, and the shared trunk goes with it: that run\n    took BCE from 0.60 to 5.99 and macro from 0.86 to 0.71. Clamping caps what\n    any one pair can demand, so a contradictory set cannot pay for scale.\n    Default None leaves every earlier run bit-identical.\n    """\n    zero = scores.sum() * 0.0\n    if ref_scores.numel() == 0:\n        return zero\n\n    gap = targets.unsqueeze(1) - ref_targets.unsqueeze(0)      # (B, Q, L)\n    pair_weight = gap.abs() * (gap.abs() >= min_gap)\n    if weights is not None and ref_weights is not None:\n        pair_weight = (pair_weight * weights.unsqueeze(1)\n                       * ref_weights.unsqueeze(0))\n    if pair_mask is not None:\n        pair_weight = pair_weight * pair_mask.to(pair_weight.dtype)\n\n    total = pair_weight.sum()\n    if total <= 0:\n        return zero\n\n    margin = scores.unsqueeze(1) - ref_scores.unsqueeze(0)     # (B, Q, L)\n    if max_margin is not None:\n        margin = margin.clamp(-max_margin, max_margin)\n    loss = F.softplus(-torch.sign(gap) * margin)\n    return (loss * pair_weight).sum() / total\n\n\ndef finding_mask(findings: Sequence[str], n_labels: int = None,\n                 device=None) -> torch.Tensor:\n    """(1, 1, L) mask selecting the findings a loss is allowed to touch.\n\n    The stratified objective is for synovitis. Applying it to all twelve\n    findings would trade away ranking on eleven that are already fine, on a\n    stratification that means nothing for them.\n    """\n    n_labels = n_labels or len(config.LABELS)\n    mask = torch.zeros(n_labels, dtype=torch.float32, device=device)\n    for name in findings:\n        mask[config.LABELS.index(name)] = 1.0\n    return mask.view(1, 1, -1)\n\n\ndef caliper_pairs(strata: torch.Tensor, ref_strata: torch.Tensor,\n                  caliper: float) -> torch.Tensor:\n    """(B, Q, 1) mask of pairs whose stratifier scores are within `caliper`.\n\n    Matching on a continuous score rather than splitting it at a threshold,\n    because the mined effusion label carries its own extraction error and a\n    hard split inherits it at the boundary. The predicted effusion score is\n    continuous, scores AUC 0.888, and is available at every training step.\n    """\n    distance = (strata.unsqueeze(1) - ref_strata.unsqueeze(0)).abs()\n    return (distance <= caliper).unsqueeze(-1)\n\n\ndef stratified_rank_loss(scores: torch.Tensor, targets: torch.Tensor,\n                         ref_scores: torch.Tensor, ref_targets: torch.Tensor,\n                         strata: torch.Tensor, ref_strata: torch.Tensor,\n                         findings: Sequence[str], caliper: float = 0.15,\n                         weights: Optional[torch.Tensor] = None,\n                         ref_weights: Optional[torch.Tensor] = None,\n                         min_gap: float = 0.25,\n                         max_margin: Optional[float] = 8.0) -> torch.Tensor:\n    """RankNet restricted to pairs from the same effusion stratum.\n\n    Conditional AUC *is* within-stratum pair concordance, so this is not an\n    analogy to the metric -- it is the metric, differentiably. Pairs the\n    conditional number ignores are the ones effusion already orders correctly,\n    and training on them is what produced a synovitis head that reconstructs\n    from the effusion head at R^2 0.55.\n\n    The known risk, stated in the plan and not solved here: stripping effusion\n    removes the one reliable cue and leaves a harder task supervised by labels\n    that are silent in 82.2% of reports against a true prevalence of 46.6%. If\n    label quality binds, this loss makes the finding worse, and the conditional\n    metric is what will show it.\n\n    `strata` is the detached predicted effusion probability for the current\n    batch and `ref_strata` the same for the queued reference studies.\n\n    `max_margin` defaults to a finite 8.0 here rather than to None, because\n    this is the loss whose pairs contradict each other -- see\n    `pairwise_rank_loss`. A margin of 8 logits is a probability ratio of about\n    3,000:1; a pair that wants more than that is not asking for an ordering.\n    """\n    mask = caliper_pairs(strata, ref_strata, caliper) * finding_mask(\n        findings, scores.shape[-1], scores.device)\n    return pairwise_rank_loss(scores, targets, ref_scores, ref_targets,\n                              weights, ref_weights, min_gap=min_gap,\n                              pair_mask=mask, max_margin=max_margin)\n\n\ndef coupling_penalty(scores: torch.Tensor, ref_scores: torch.Tensor,\n                     child: int, parent: int, target: float) -> torch.Tensor:\n    """Push corr(child logit, parent logit) toward `target`, not toward zero.\n\n    The findings genuinely co-occur: on the gold set the odds ratio is 6.09\n    (p = 0.003) and the label correlation is 0.403, against the model\'s 0.651.\n    Driving the two heads to independence optimises toward a value the truth\n    contradicts, so `target` is required and is meant to be the observed\n    coupling rather than 0.\n\n    Batches here are one study per rank, so the correlation is computed over\n    the current batch concatenated with the detached reference queue. Returns\n    a zero that still touches `scores` when there are too few studies to\n    estimate a correlation, so the graph does not change shape between steps.\n    """\n    zero = scores.sum() * 0.0\n    pool = (torch.cat([scores, ref_scores], dim=0)\n            if ref_scores.numel() else scores)\n    if pool.shape[0] < 8:\n        return zero\n\n    a = pool[:, child] - pool[:, child].mean()\n    b = pool[:, parent] - pool[:, parent].mean()\n    denominator = a.norm() * b.norm()\n    if denominator <= 0:\n        return zero\n    return ((a * b).sum() / denominator - target) ** 2\n\n\ndef load_knee_state(path, map_location="cpu") -> dict:\n    """Read a `KneeNet` state dict from either checkpoint format.\n\n    The end-to-end path writes two things over its life: a bare state dict\n    from `torch.save(model.state_dict(), ...)`, and a Lightning checkpoint,\n    which nests the same tensors under `state_dict` and prefixes every key\n    with `model.` because `KneeNet` is an attribute of the wrapper. The\n    submission must not care which it was handed -- requiring a manual\n    conversion step is requiring a step that can be forgotten, and it was.\n\n    Lightning checkpoints also carry optimiser state and callback objects,\n    which `weights_only=True` refuses, so that is attempted first and relaxed\n    only on failure. These are files this project wrote.\n    """\n    try:\n        obj = torch.load(path, map_location=map_location, weights_only=True)\n    except TypeError:       # torch too old to know the argument\n        obj = torch.load(path, map_location=map_location)\n    except Exception:       # noqa: BLE001 - a Lightning checkpoint\n        obj = torch.load(path, map_location=map_location, weights_only=False)\n\n    state = obj.get("state_dict", obj) if isinstance(obj, dict) else obj\n    if any(key.startswith("model.") for key in state):\n        state = {key[len("model."):]: value for key, value in state.items()\n                 if key.startswith("model.")}\n    if not state:\n        raise ValueError(f"no weights found in {path}")\n    return state\n',
    'src/model/conditional.py': '"""Effusion-stratified conditional AUC: the metric synovitis is judged on.\n\nPooled AUC cannot measure this finding. On the 58 gold studies the *gold\neffusion label by itself* scores 0.698 as a synovitis classifier, against the\nsynovitis head\'s 0.701 -- so a number in that region is obtainable with no\nsynovitis-specific skill at all, and seven consecutive experiments read as\nnull partly because they were scored on it.\n\nConditional AUC is the concordance of positive/negative pairs drawn from\n**within the same effusion stratum**. Cross-stratum pairs, where effusion\nalone orders the pair correctly, are dropped. What remains is the only\nquantity on this gold set that moves when synovitis-specific skill moves:\n\n    pooled       0.701      837 pairs, ~55% of them cross-stratum\n    conditional  0.630      376 within-stratum pairs\n\nThe two are linked by pair mass, which is why `pooled_from_conditional` can\nstate the exact pooled number a conditional target implies on this set --\nconditional 0.75 is pooled 0.755, and a *perfect* conditional score is pooled\n0.867. Anything above that is unreachable without changing the gating.\n\nDeliberately free of torch and of pandas-at-import: this module is imported by\nthe analysis tools, bootstrapped into the notebooks, and used by the training\nloop\'s validation hook, and `src/model/lightning.py` cannot be imported by any\nof the first two without dragging in the Lightning framework.\n\nTwo rules this module exists to enforce, from the plan\'s standing rules:\n\n1. Never judge synovitis on macro, and never on pooled alone.\n2. Compare paired. Independent Hanley-McNeil standard errors are +/-0.07 here\n   and carry between-study variance that a matched comparison removes; a real\n   movement of 0.05 is invisible against them and visible against a paired\n   bootstrap of the difference.\n"""\n\nfrom typing import Dict, Optional, Sequence\n\nimport numpy as np\n\n\n# ---------------------------------------------------------------------------\n# Pooled AUC, and the noise band around it\n# ---------------------------------------------------------------------------\n\ndef auc(y_true, scores) -> float:\n    """ROC AUC via the rank statistic, ties averaged. Matches the training loop."""\n    y_true = np.asarray(y_true)\n    scores = np.asarray(scores, dtype=float)\n    positives = y_true == 1\n    n_pos, n_neg = int(positives.sum()), int((~positives).sum())\n    if n_pos == 0 or n_neg == 0:\n        return float("nan")\n\n    order = np.argsort(scores, kind="mergesort")\n    ranks = np.empty(len(scores), dtype=float)\n    ranks[order] = np.arange(1, len(scores) + 1)\n    unique, inverse, counts = np.unique(scores, return_inverse=True,\n                                        return_counts=True)\n    sums = np.zeros(len(unique))\n    np.add.at(sums, inverse, ranks)\n    ranks = (sums / counts)[inverse]\n    return float((ranks[positives].sum() - n_pos * (n_pos + 1) / 2)\n                 / (n_pos * n_neg))\n\n\ndef auc_se(a: float, n_pos: int, n_neg: int) -> float:\n    """Hanley & McNeil (1982) standard error of an AUC.\n\n    Printed beside pooled numbers so a difference inside the band is visibly\n    not evidence. Never used to judge a *difference* between two models on the\n    same studies -- that is what `paired_bootstrap` is for.\n    """\n    if not np.isfinite(a) or n_pos < 1 or n_neg < 1:\n        return float("nan")\n    q1 = a / (2.0 - a)\n    q2 = 2.0 * a * a / (1.0 + a)\n    numerator = (a * (1 - a) + (n_pos - 1) * (q1 - a * a)\n                 + (n_neg - 1) * (q2 - a * a))\n    return float(np.sqrt(max(numerator, 0.0) / (n_pos * n_neg)))\n\n\n# ---------------------------------------------------------------------------\n# Strata\n# ---------------------------------------------------------------------------\n\ndef strata_from_scores(effusion, threshold: Optional[float] = None):\n    """Binary effusion strata from a continuous effusion score.\n\n    The plan stratifies on *predicted* effusion rather than on the mined\n    effusion label, because the prediction scores AUC 0.888 while the mined\n    label carries its own extraction error that would blur the strata. The\n    default split is the median, which is what reproduces the plan\'s 0.617\n    against the gold label\'s 0.630 -- close enough that the metric is\n    computable during training, where no gold effusion label exists.\n\n    A median split rather than a fixed probability because the head is not\n    calibrated across runs: a 0.70 cut that balances one model\'s studies\n    puts 80% of another model\'s on one side, and the number would then move\n    between runs for a reason that has nothing to do with synovitis.\n    """\n    effusion = np.asarray(effusion, dtype=float)\n    if threshold is None:\n        threshold = float(np.median(effusion))\n    return (effusion >= threshold).astype(int)\n\n\ndef comparable_pairs(y_true, strata, caliper: Optional[float] = None):\n    """(n_pos, n_neg) boolean mask of pairs that share an effusion stratum.\n\n    With `caliper=None`, `strata` is treated as categorical and pairs must\n    match exactly -- the gold-label and median-split cases. With a caliper,\n    `strata` is a continuous score and a pair is comparable when the two\n    studies\' effusion scores lie within `caliper` of each other, which is the\n    matching rule the stratified training loss uses.\n    """\n    y_true = np.asarray(y_true)\n    strata = np.asarray(strata)\n    positives = y_true == 1\n    left, right = strata[positives], strata[~positives]\n    if caliper is None:\n        return left[:, None] == right[None, :]\n    return np.abs(left[:, None].astype(float)\n                  - right[None, :].astype(float)) <= caliper\n\n\ndef _concordance(y_true, scores):\n    """(n_pos, n_neg) matrix of 1 / 0.5 / 0 per pair. Ties count as half."""\n    y_true = np.asarray(y_true)\n    scores = np.asarray(scores, dtype=float)\n    positives = y_true == 1\n    difference = scores[positives][:, None] - scores[~positives][None, :]\n    return (difference > 0).astype(float) + 0.5 * (difference == 0)\n\n\ndef conditional_auc(y_true, scores, strata,\n                    caliper: Optional[float] = None) -> float:\n    """Concordance over within-stratum pairs only. NaN if there are none.\n\n    Pair-weighted, not stratum-averaged: the strata are 286 and 90 pairs on\n    the gold set, and averaging them would give the 5-positive stratum equal\n    say with the 22-positive one. Pair weighting is also what makes the\n    conversion to pooled AUC exact.\n    """\n    mask = comparable_pairs(y_true, strata, caliper)\n    total = mask.sum()\n    if total == 0:\n        return float("nan")\n    return float((_concordance(y_true, scores) * mask).sum() / total)\n\n\ndef pair_counts(y_true, strata, caliper: Optional[float] = None) -> Dict[str, int]:\n    """How many pairs the conditional number rests on, and the total."""\n    mask = comparable_pairs(y_true, strata, caliper)\n    return {"within": int(mask.sum()), "cross": int((~mask).sum()),\n            "total": int(mask.size)}\n\n\n# ---------------------------------------------------------------------------\n# Where the error sits\n# ---------------------------------------------------------------------------\n\ndef pair_decomposition(y_true, scores, strata) -> list:\n    """Every discordant pair, split by the stratum of each member.\n\n    The table in the plan\'s §1.4. Reading it is how the addressable headroom\n    was located: within-stratum pairs are 44.9% of the total and are the only\n    ones a synovitis-specific improvement can move. The cell where a\n    stratum-0 positive meets a stratum-1 negative is ranked worse than chance\n    -- that is the gating tax, and it is paid on five studies.\n    """\n    y_true = np.asarray(y_true)\n    strata = np.asarray(strata)\n    positives = y_true == 1\n    concordance = _concordance(y_true, scores)\n    error = 1.0 - concordance\n    total_error = error.sum()\n\n    rows = []\n    for pos_level in sorted(set(strata[positives].tolist()), reverse=True):\n        for neg_level in sorted(set(strata[~positives].tolist()), reverse=True):\n            mask = ((strata[positives] == pos_level)[:, None]\n                    & (strata[~positives] == neg_level)[None, :])\n            pairs = int(mask.sum())\n            if pairs == 0:\n                continue\n            cell_error = float((error * mask).sum())\n            rows.append({\n                "pos_stratum": int(pos_level), "neg_stratum": int(neg_level),\n                "within": pos_level == neg_level,\n                "pairs": pairs,\n                "misranked": cell_error / pairs,\n                "error_share": cell_error / total_error if total_error else float("nan"),\n            })\n    return rows\n\n\ndef pooled_from_conditional(y_true, scores, strata,\n                            targets: Sequence[float]) -> list:\n    """The pooled AUC each conditional target implies, on these studies.\n\n    Cross-stratum concordance is held at what this model already achieves and\n    only the within-stratum mass is moved, so the answer is arithmetic rather\n    than a fit. It exists to keep two conversations in the same units: the\n    leaderboard and every historical table are pooled, the plan\'s targets are\n    conditional, and quoting one as the other is how a 0.63 -> 0.75 move gets\n    dismissed as "+0.05, inside the noise".\n\n    The value at `target=1.0` is the ceiling under an unchanged gating -- 0.867\n    here. It is an arithmetic bound, not a reachable target: one model produces\n    both halves, and a residualised model can buy within-stratum ordering by\n    selling the gating.\n    """\n    mask = comparable_pairs(y_true, strata)\n    concordance = _concordance(y_true, scores)\n    within, cross = mask.sum(), (~mask).sum()\n    cross_concordant = float((concordance * ~mask).sum())\n    total = within + cross\n    return [{"conditional": float(t),\n             "pooled": float((cross_concordant + t * within) / total)}\n            for t in targets]\n\n\n# ---------------------------------------------------------------------------\n# Uncertainty: paired, always\n# ---------------------------------------------------------------------------\n\ndef _resamples(n: int, draws: int, seed: int):\n    rng = np.random.default_rng(seed)\n    return rng.integers(0, n, size=(draws, n))\n\n\ndef bootstrap_conditional(y_true, scores, strata, caliper=None,\n                          draws: int = 4000, seed: int = 0) -> Dict[str, float]:\n    """Percentile CI for one model\'s conditional AUC, plus P(AUC <= 0.5).\n\n    Studies are resampled, not pairs: a study appears in many pairs and\n    resampling pairs would treat those as independent observations and report\n    an interval far narrower than the evidence supports.\n\n    `p_at_or_below_chance` is the fraction of resamples at or under 0.5. It is\n    the number the gate is read on, because "is there any synovitis-specific\n    signal at all" is a question about which side of chance the estimate sits,\n    not about whether it differs from some other model.\n    """\n    y_true = np.asarray(y_true)\n    scores = np.asarray(scores, dtype=float)\n    strata = np.asarray(strata)\n\n    values = []\n    for index in _resamples(len(y_true), draws, seed):\n        value = conditional_auc(y_true[index], scores[index], strata[index],\n                                caliper)\n        if np.isfinite(value):\n            values.append(value)\n    values = np.asarray(values)\n    point = conditional_auc(y_true, scores, strata, caliper)\n    if values.size == 0:\n        return {"auc": point, "lo": float("nan"), "hi": float("nan"),\n                "sd": float("nan"), "p_at_or_below_chance": float("nan"),\n                "draws": 0}\n    return {"auc": point,\n            "lo": float(np.percentile(values, 2.5)),\n            "hi": float(np.percentile(values, 97.5)),\n            "sd": float(values.std(ddof=1)),\n            "p_at_or_below_chance": float((values <= 0.5).mean()),\n            "draws": int(values.size)}\n\n\ndef paired_bootstrap(y_true, scores_a, scores_b, strata=None, caliper=None,\n                     draws: int = 4000, seed: int = 0) -> Dict[str, float]:\n    """CI for (A - B) on the same studies, conditional or pooled.\n\n    `strata=None` compares pooled AUCs; otherwise both models are scored on\n    the same within-stratum pairs. Either way the *same* resample indexes both\n    models, so between-study variance -- which is most of the +/-0.07 an\n    unpaired standard error reports -- cancels.\n\n    `p_a_not_better` is the fraction of resamples where the difference is at\n    or below zero. Read it as the evidence against A, and pre-register the\n    threshold before running, not after seeing it.\n    """\n    y_true = np.asarray(y_true)\n    scores_a = np.asarray(scores_a, dtype=float)\n    scores_b = np.asarray(scores_b, dtype=float)\n    strata = None if strata is None else np.asarray(strata)\n\n    def statistic(y, a, b, s):\n        if s is None:\n            return auc(y, a) - auc(y, b)\n        return (conditional_auc(y, a, s, caliper)\n                - conditional_auc(y, b, s, caliper))\n\n    values = []\n    for index in _resamples(len(y_true), draws, seed):\n        value = statistic(y_true[index], scores_a[index], scores_b[index],\n                          None if strata is None else strata[index])\n        if np.isfinite(value):\n            values.append(value)\n    values = np.asarray(values)\n    point = statistic(y_true, scores_a, scores_b, strata)\n    if values.size == 0:\n        return {"delta": point, "lo": float("nan"), "hi": float("nan"),\n                "p_a_not_better": float("nan"), "draws": 0}\n    return {"delta": point,\n            "lo": float(np.percentile(values, 2.5)),\n            "hi": float(np.percentile(values, 97.5)),\n            "p_a_not_better": float((values <= 0).mean()),\n            "draws": int(values.size)}\n\n\n# ---------------------------------------------------------------------------\n# Calibrating the residual coupling\n# ---------------------------------------------------------------------------\n\ndef logit(probability, floor: float = 1e-6):\n    """Probabilities to logits, clipped off the asymptotes."""\n    clipped = np.clip(np.asarray(probability, dtype=float), floor, 1 - floor)\n    return np.log(clipped / (1 - clipped))\n\n\ndef calibrate_coupling(labels, parent_logits, iterations: int = 50) -> dict:\n    """Fit `P(child=1) = sigmoid(a + b * parent_logit)` and return b.\n\n    This is the beta for the residual parameterisation\n    `child_logit = beta * stopgrad(parent_logit) + g(features)`, and where it\n    comes from matters more than its value:\n\n    - **Not** from the model\'s own +0.843 reconstruction coefficient. That is\n      the coupling the current model already has, and reproducing it by\n      construction would hand the residual head exactly the term it is\n      supposed to replace.\n    - **Not** from r_true = 0.403. A rank correlation between two binary\n      labels does not map onto a logit coefficient, and the plan says so\n      explicitly.\n\n    It is fitted out of fold, on labels, by IRLS -- the coupling the *truth*\n    supports between the two findings, measured on whatever label set is\n    trustworthy enough to fit it. `separated` flags the degenerate case where\n    the fit runs away because the parent perfectly orders the child; the\n    coefficient is then meaningless and should not be used.\n    """\n    y = np.asarray(labels, dtype=float)\n    x = np.asarray(parent_logits, dtype=float)\n    ok = np.isfinite(y) & np.isfinite(x)\n    y, x = y[ok], x[ok]\n    if y.size < 10 or y.std() == 0:\n        return {"beta": float("nan"), "intercept": float("nan"),\n                "n": int(y.size), "separated": True}\n\n    design = np.column_stack([np.ones_like(x), x])\n    coefficients = np.zeros(2)\n    for _ in range(iterations):\n        # Clipped before the exponential: under separation the coefficients\n        # run away and the naive form overflows on the way to being flagged.\n        eta = np.clip(design @ coefficients, -700, 700)\n        mu = 1.0 / (1.0 + np.exp(-eta))\n        weight = np.clip(mu * (1 - mu), 1e-9, None)\n        gradient = design.T @ (y - mu)\n        hessian = design.T @ (design * weight[:, None])\n        try:\n            step = np.linalg.solve(hessian, gradient)\n        except np.linalg.LinAlgError:\n            return {"beta": float("nan"), "intercept": float("nan"),\n                    "n": int(y.size), "separated": True}\n        coefficients = coefficients + step\n        if np.max(np.abs(step)) < 1e-8:\n            break\n    separated = bool(np.max(np.abs(coefficients)) > 20)\n    return {"beta": float(coefficients[1]), "intercept": float(coefficients[0]),\n            "n": int(y.size), "separated": separated}\n\n\n# ---------------------------------------------------------------------------\n# The Stage 0 gate\n# ---------------------------------------------------------------------------\n#\n# The plan says "clearly > 0.5", "= 0.5" and "ambiguous" without giving\n# numbers. They are fixed here, once, in code, so they cannot be chosen after\n# seeing a result -- and here rather than in the tool because the training\n# notebooks read the same gate and two copies would drift.\n\nGATE_ACTIONS = {\n    "clear": ("synovitis-specific signal survives conditioning on effusion",\n              "Stage 1 -- train the conditional task directly"),\n    "chance": ("the score is a better effusion proxy, nothing more",\n               "Stage 3 (plan §4) or Stage 4 descope (plan §10) -- the plan "\n               "gives both; decide before running either"),\n    "ambiguous": ("expected at n=58; the CI straddles chance",\n                  "Stage 1 at reduced confidence, and prioritise Stage 2 "\n                  "(expand the gold set)"),\n}\n\n\ndef gate_reading(stats: dict) -> str:\n    """clear | chance | ambiguous, from a `bootstrap_conditional` result.\n\n        clear      2.5th percentile above 0.5, and P(AUC <= 0.5) < 0.05\n        chance     point estimate <= 0.55 and P(AUC <= 0.5) >= 0.25\n        ambiguous  anything else -- the expected outcome at n=58\n    """\n    if not np.isfinite(stats.get("auc", float("nan"))):\n        return "ambiguous"\n    if stats["lo"] > 0.5 and stats["p_at_or_below_chance"] < 0.05:\n        return "clear"\n    if stats["auc"] <= 0.55 and stats["p_at_or_below_chance"] >= 0.25:\n        return "chance"\n    return "ambiguous"\n',
    'src/model/augment.py': '"""Training augmentation for study volumes.\n\nThe pipeline shipped with one augmentation -- a single intensity scale in\n[0.9, 1.1] applied to a whole study -- while the plan (§3.4) calls for what the\nRSNA winners used: geometric transforms, photometric jitter, noise, region\ndropout, and slice-order permutation, all kept consistent across the slices of\na series.\n\n**Series-consistent by construction.** Every transform draws its parameters\nonce per series and applies them to all of that series\' slices. The alternative\n-- fresh parameters per slice -- turns a coherent volume into a stack of\nunrelated images, which is exactly the through-plane structure `stack_2p5d`\nexists to give the backbone.\n\n**What is deliberately absent, and why:**\n\n- **Rotation, perspective, affine.** They need `scipy.ndimage` or `cv2` and run\n  per-slice interpolation over ~80 slices per study; at 4,349 studies an epoch\n  that is a dataloader bottleneck, not a free regulariser. Worth adding behind\n  a measurement of loader throughput, not before.\n- **Arbitrary slice permutation.** The plan lists it, but it would break the\n  2.5D stacking: `stack_2p5d` gives each slice its two neighbours as channels,\n  and a shuffled stack makes those neighbours meaningless. `slice_reverse`\n  is the order change that keeps neighbour structure intact -- it reverses the\n  through-plane direction rather than destroying it.\n- **MixUp.** Not implementable at `BATCH_STUDIES = 1`: a within-batch\n  permutation is the identity. Recorded in `docs/plan.md`.\n\n**Laterality flip is a study-level transform, not a series-level one.** All of\na study\'s series image the same knee, so they must flip together, and the flip\nswaps medial with lateral -- which means `config.LATERAL_LABEL_PAIRS` has to be\nswapped in the targets at the same time. `config.py` has carried that constant\nand that warning since before there was anything to use it. Getting this half\nright (flipping pixels, not labels) corrupts four of the twelve findings\nsilently, which is why the dataset refused to flip at all until now.\n"""\n\nfrom dataclasses import dataclass\nfrom typing import Optional, Tuple\n\nimport numpy as np\n\nfrom src import config\nfrom src.model.dataset import (DEFAULT_MIRROR_AXIS, META_COLUMNS,\n                               PLANE_MIRROR_AXIS)\n\n# Index of `is_sagittal` in the per-series metadata row. The flip axis depends\n# on the acquisition plane, and this is where that is already encoded -- no\n# need to reach back into the index for `Anatomical_Plane`.\nSAGITTAL_COLUMN = META_COLUMNS.index("is_sagittal")\n\n\ndef flip_volume(volume: np.ndarray, plane=None,\n                sagittal: Optional[bool] = None) -> np.ndarray:\n    """Mirror one series across the medial-lateral axis.\n\n    The axis comes from `dataset.PLANE_MIRROR_AXIS`, the same table\n    `mirror_volume` uses to canonicalise laterality -- deliberately, because a\n    training flip and a test-time flip that disagreed about which axis is\n    medial-lateral would be a mismatch that runs clean and scores worse. On a\n    sagittal series medial-lateral is the slice axis; on coronal and axial it\n    lies in-plane.\n\n    Give it either `plane` (the string from the series metadata) or `sagittal`\n    (the bit already encoded in a metadata row).\n    """\n    if sagittal is not None:\n        axis = PLANE_MIRROR_AXIS["Sagittal"] if sagittal else DEFAULT_MIRROR_AXIS\n    else:\n        axis = PLANE_MIRROR_AXIS.get(str(plane), DEFAULT_MIRROR_AXIS)\n    return np.flip(volume, axis=axis).copy()\n\n\ndef lateral_swap_order():\n    """Index permutation that exchanges the medial/lateral finding pairs.\n\n    A mirrored knee puts the medial compartment where the lateral one was, so\n    any prediction or target read off a flipped image has to be permuted by\n    this before it means anything. Used by the training augmentation and by\n    the submission\'s flip TTA, from one definition, so the two cannot drift.\n    """\n    order = list(range(len(config.LABELS)))\n    index_of = {name: i for i, name in enumerate(config.LABELS)}\n    for medial, lateral in config.LATERAL_LABEL_PAIRS:\n        a, b = index_of[medial], index_of[lateral]\n        order[a], order[b] = order[b], order[a]\n    return order\n\n\n@dataclass(frozen=True)\nclass AugmentConfig:\n    """Which augmentations run, and how hard.\n\n    The defaults reproduce the incumbent behaviour exactly -- one intensity\n    scale, nothing else -- so `AugmentConfig()` is the reference configuration\n    and every measured result stays comparable. Turn things on one at a time.\n    """\n\n    intensity_scale: Optional[Tuple[float, float]] = (0.9, 1.1)\n    intensity_shift: float = 0.0        # +/- this many 8-bit levels\n    gamma: Optional[Tuple[float, float]] = None\n    noise_std: float = 0.0              # gaussian sigma, in 8-bit levels\n    cutout_boxes: int = 0\n    cutout_fraction: float = 0.25       # box side, as a fraction of the image\n    slice_reverse: float = 0.0          # probability, per series\n    laterality_flip: float = 0.0        # probability, per study\n\n    def __post_init__(self):\n        if not 0.0 <= self.slice_reverse <= 1.0:\n            raise ValueError("slice_reverse must be a probability")\n        if not 0.0 <= self.laterality_flip <= 1.0:\n            raise ValueError("laterality_flip must be a probability")\n        if not 0.0 < self.cutout_fraction <= 1.0:\n            raise ValueError("cutout_fraction must be in (0, 1]")\n\n\nREFERENCE = AugmentConfig()\n\n\ndef augment_series(volume: np.ndarray, rng: np.random.Generator,\n                   cfg: AugmentConfig = REFERENCE) -> np.ndarray:\n    """(D, H, W) uint8 -> (D, H, W) uint8, one parameter draw for the series.\n\n    Photometric work happens in float32 and is clipped back to uint8 once at\n    the end, so a scale followed by a shift does not round twice.\n    """\n    working = volume.astype(np.float32)\n\n    if cfg.intensity_scale is not None:\n        low, high = cfg.intensity_scale\n        working *= rng.uniform(low, high)\n\n    if cfg.intensity_shift:\n        working += rng.uniform(-cfg.intensity_shift, cfg.intensity_shift)\n\n    if cfg.gamma is not None:\n        low, high = cfg.gamma\n        # Gamma is defined on [0, 1]; the volume is 8-bit, so scale in and out.\n        working = np.clip(working, 0, 255)\n        working = 255.0 * np.power(working / 255.0, rng.uniform(low, high))\n\n    if cfg.noise_std > 0:\n        # One noise field for the whole series rather than per slice: the\n        # sensor noise being imitated is spatially structured, and per-slice\n        # noise is trivially averaged away by the slice pooling.\n        working = working + rng.normal(0.0, cfg.noise_std,\n                                       size=working.shape[1:])[None]\n\n    if cfg.cutout_boxes > 0:\n        depth, height, width = working.shape\n        box_h = max(1, int(round(height * cfg.cutout_fraction)))\n        box_w = max(1, int(round(width * cfg.cutout_fraction)))\n        for _ in range(cfg.cutout_boxes):\n            top = int(rng.integers(0, max(1, height - box_h + 1)))\n            left = int(rng.integers(0, max(1, width - box_w + 1)))\n            # The SAME box on every slice: a box that moves through the stack\n            # is a different occlusion per slice, which the slice pooling can\n            # simply average out instead of learning around.\n            working[:, top:top + box_h, left:left + box_w] = 0.0\n\n    if cfg.slice_reverse and rng.random() < cfg.slice_reverse:\n        working = working[::-1]\n\n    return np.clip(working, 0, 255).astype(np.uint8)\n\n\ndef flip_laterality(item: dict, rng: np.random.Generator) -> dict:\n    """Mirror every series of a study and swap the medial/lateral labels.\n\n    Volumes reach here already canonicalised to the right knee, so this maps\n    the study onto the left and back-labels it accordingly -- doubling the\n    laterality distribution the model sees without inventing anatomy.\n\n    The flip axis depends on the acquisition plane, exactly as\n    `dataset.mirror_volume` decides it: on a sagittal series left-right runs\n    *through* the plane, so mirroring reverses the slice order; on coronal and\n    axial it lies in-plane, so it flips the columns. `is_sagittal` in the\n    metadata row is what distinguishes them.\n\n    Targets and weights are permuted by `config.LATERAL_LABEL_PAIRS`. Both, not\n    just the targets: a weight says how much a finding\'s label is believed, and\n    it belongs to the finding, so it has to travel with it.\n    """\n    volumes, meta = item["volumes"], item["meta"]\n    flipped = np.empty_like(volumes)\n    for i in range(volumes.shape[0]):\n        flipped[i] = flip_volume(\n            volumes[i], sagittal=bool(meta[i, SAGITTAL_COLUMN] > 0.5))\n    item["volumes"] = flipped\n\n    order = lateral_swap_order()\n    for key in ("targets", "weights"):\n        if item.get(key) is not None:\n            item[key] = np.asarray(item[key])[order]\n\n    # `is_right` in the metadata now describes the wrong side. Left as it is on\n    # purpose: it is a *prediction* from the side classifier describing the\n    # image as acquired, and the model reads it as conditioning rather than as\n    # a label. Rewriting it here would tell the model the flip happened, which\n    # is precisely the invariance the augmentation exists to teach.\n    return item\n\n\ndef augment_study(item: dict, rng: np.random.Generator,\n                  cfg: AugmentConfig = REFERENCE) -> dict:\n    """Apply the study-level flip, then per-series photometric work."""\n    if cfg.laterality_flip and rng.random() < cfg.laterality_flip:\n        item = flip_laterality(item, rng)\n\n    volumes = item["volumes"]\n    if volumes.shape[0]:\n        item["volumes"] = np.stack(\n            [augment_series(volumes[i], rng, cfg)\n             for i in range(volumes.shape[0])])\n    return item\n',
    'src/model/averaging.py': '"""Weight averaging: EMA, SWA, and model soups.\n\n**Why this module exists.** Checkpoint selection here is broken in a specific,\ndiagnosable way. Training labels are mined from radiology reports and agree\nwith image-derived ground truth about 82% of the time, and the validation fold\nis mined from the *same* distribution. So the network first fits the structure\nthe two sources share and then memorises the 18% they do not -- Arpit et al.,\n"A Closer Look at Memorization in Deep Networks" (ICML 2017), and Liu et al.,\n"Early-Learning Regularization Prevents Memorization of Noisy Labels" (NeurIPS\n2020). Mined validation agreement keeps *rising* through that second phase,\nbecause agreeing with the noise is what it measures. It is therefore a biased\nselector, not merely a noisy one, and "take the best epoch on validation"\nselects into the memorisation phase on purpose.\n\nMeasured here: mined val rose to 0.860 at epoch 9 and fell to 0.843 by epoch\n20, while gold held flat at 0.864-0.869 across eleven consecutive epochs.\n\nThe fix is to stop selecting. Averaging weights is selection-free:\n\n- **EMA** (Tarvainen & Valpola, NeurIPS 2017) -- an exponential moving average\n  maintained during training. Cheap, standard, and already the default in most\n  detection and segmentation pipelines.\n- **SWA** (Izmailov et al., UAI 2018, arXiv:1803.05407) -- an equal-weight\n  average over the tail of the trajectory, which lands in a flatter and better\n  generalising region than any single point on it.\n- **Model soups** (Wortsman et al., ICML 2022, arXiv:2203.05482) -- average\n  the weights of several finished fine-tunes instead of picking the best one\n  on a held-out set. The paper\'s argument is directed at exactly the recipe\n  failing here: "pick the individual model which performs best on a held-out\n  validation set, discarding the remainder".\n\n**Averaging weights is only meaningful for models in the same basin.** Two\nruns whose aggregation was randomly initialised from different seeds are not,\nand averaging them produces a model that is worse than either -- silently, as\na plausible-looking set of numbers. `assert_soupable` checks what it can\n(shapes and key sets); it cannot check the basin, so the training notebook\nseeds every fold identically, and the docstrings below say when a soup is\nwell-posed.\n\n**Nothing here may select on the gold 58.** `greedy_soup` takes a caller\'s\nscore function precisely so the choice of scoring set stays visible at the\ncall site: it must be mined validation or an out-of-fold dump. Selecting a\nsoup on gold would turn the project\'s one honest measuring stick into a\nmodel-selection set, which has already cost this project an inflated figure\nonce.\n"""\n\nfrom typing import Callable, Dict, List, Optional, Sequence, Tuple\n\nimport torch\n\nState = Dict[str, torch.Tensor]\n\n\ndef _is_averageable(tensor: torch.Tensor) -> bool:\n    """Floating-point tensors average; integer counters and flags do not.\n\n    `num_batches_tracked` on a BatchNorm is the canonical example: averaging\n    it produces a fractional count, and torch stores it as int64 anyway. The\n    aggregation stack is all LayerNorm so this does not arise today, but a\n    convnet backbone with BatchNorm would hit it immediately.\n    """\n    return tensor.is_floating_point()\n\n\ndef assert_soupable(states: Sequence[State]) -> None:\n    """Refuse states that cannot be averaged elementwise.\n\n    Catches the two mechanical failures -- a different architecture, and a\n    checkpoint saved at a different width -- before they become a tensor of\n    the wrong shape several cells later.\n    """\n    if not states:\n        raise ValueError("no states to average")\n    reference = states[0]\n    for i, state in enumerate(states[1:], start=1):\n        if set(state) != set(reference):\n            missing = sorted(set(reference) - set(state))[:3]\n            extra = sorted(set(state) - set(reference))[:3]\n            raise ValueError(\n                f"state {i} has different keys from state 0 "\n                f"(missing {missing}, extra {extra}) -- these are not the "\n                "same architecture")\n        for key, tensor in state.items():\n            if tensor.shape != reference[key].shape:\n                raise ValueError(\n                    f"state {i} has {key} at {tuple(tensor.shape)} against "\n                    f"{tuple(reference[key].shape)} in state 0")\n\n\ndef average_states(states: Sequence[State],\n                   weights: Optional[Sequence[float]] = None) -> State:\n    """Elementwise weighted average of state dicts.\n\n    `weights` defaults to equal, which is SWA and the uniform soup. Averaging\n    happens in float64 and is cast back to each tensor\'s own dtype, so a run\n    of fp16 checkpoints does not accumulate rounding across a dozen members.\n\n    Non-floating tensors are taken from the first state rather than averaged.\n    """\n    assert_soupable(states)\n    if weights is None:\n        weights = [1.0] * len(states)\n    if len(weights) != len(states):\n        raise ValueError(f"{len(weights)} weights for {len(states)} states")\n    total = float(sum(weights))\n    if total <= 0:\n        raise ValueError("weights must sum to something positive")\n\n    averaged: State = {}\n    for key, reference in states[0].items():\n        if not _is_averageable(reference):\n            averaged[key] = reference.clone()\n            continue\n        accumulator = torch.zeros_like(reference, dtype=torch.float64)\n        for state, weight in zip(states, weights):\n            accumulator += state[key].to(torch.float64) * float(weight)\n        averaged[key] = (accumulator / total).to(reference.dtype)\n    return averaged\n\n\nclass WeightAverage:\n    """EMA and SWA of one model\'s weights, maintained during training.\n\n    Both are accumulated from the same call so a run pays one copy of the\n    weights per update and produces both artefacts. `decay` is the EMA\'s, and\n    0.999 is the usual choice; `swa_from` is the step at which the equal-weight\n    tail average starts, so a caller sets it to 75% of the planned steps to\n    average over the last quarter.\n\n    **The averages live wherever the model does, in float32.** Accumulating on\n    the host instead would copy the whole trunk across the PCIe bus on every\n    optimiser step -- ~110 MB for ConvNeXt-T plus the aggregation, twice per\n    step once the SWA window opens, about 11,000 times over a twenty-epoch\n    run. That is a wall-clock tax on the run for no benefit; two shadows on\n    the device cost ~220 MB of a 16 GB T4, which the batch has to spare.\n    Float32 regardless of training precision: under `16-mixed` the parameters\n    are fp32 anyway (autocast does not change them), but an fp16 accumulation\n    over thousands of steps would round away exactly the small updates an EMA\n    exists to keep. `export` moves the result to the host.\n\n    `device` overrides the placement -- pass "cpu" if a run is ever tight\n    enough on device memory that 220 MB matters.\n    """\n\n    def __init__(self, model: torch.nn.Module, decay: float = 0.999,\n                 swa_from: Optional[int] = None, device=None):\n        if not 0.0 < decay < 1.0:\n            raise ValueError(f"decay must be in (0, 1), got {decay}")\n        self.decay = decay\n        self.swa_from = swa_from\n        self.steps = 0\n        self.swa_count = 0\n        state = model.state_dict()\n        self.ema: State = {\n            k: (v.detach().clone().to(torch.float32) if device is None\n                else v.detach().to(device, torch.float32).clone())\n            for k, v in state.items() if _is_averageable(v)}\n        self.swa: State = {k: torch.zeros_like(v) for k, v in self.ema.items()}\n        # Everything not averaged -- integer counters. The normalisation\n        # buffers are float, so they average harmlessly to themselves.\n        self._fixed: State = {k: v.detach().cpu().clone()\n                              for k, v in state.items()\n                              if not _is_averageable(v)}\n\n    @torch.no_grad()\n    def update(self, model: torch.nn.Module) -> None:\n        """Fold one optimiser step\'s weights into both averages."""\n        self.steps += 1\n        state = model.state_dict()\n        in_window = (self.swa_from is not None and self.steps >= self.swa_from)\n        if in_window:\n            self.swa_count += 1\n\n        for key, shadow in self.ema.items():\n            # Fetched once and reused by both averages: reading it per average\n            # doubled the copy for every step of the SWA window.\n            current = state[key].detach().to(shadow.device, torch.float32,\n                                             non_blocking=True)\n            shadow.mul_(self.decay).add_(current, alpha=1.0 - self.decay)\n            if in_window:\n                accumulator = self.swa[key]\n                # Running mean, so the tail can be any length without a second\n                # pass or a list of checkpoints held in memory.\n                accumulator.add_((current - accumulator) / self.swa_count)\n\n    def ema_state(self) -> State:\n        """The EMA weights, as a state dict a `KneeNet` loads.\n\n        Moved to the host on the way out: a checkpoint holding CUDA tensors\n        reloads onto a device that may not exist in the process reading it.\n        """\n        return {**self._fixed, **{k: v.detach().cpu().clone()\n                                  for k, v in self.ema.items()}}\n\n    def swa_state(self) -> Optional[State]:\n        """The tail average, or None if no step fell inside the window."""\n        if self.swa_count == 0:\n            return None\n        return {**self._fixed, **{k: v.detach().cpu().clone()\n                                  for k, v in self.swa.items()}}\n\n\ndef greedy_soup(states: Sequence[State], score: Callable[[State], float],\n                names: Optional[Sequence[str]] = None,\n                tolerance: float = 0.0) -> Tuple[State, List[str]]:\n    """Wortsman et al.\'s greedy soup: add a member only if it helps.\n\n    Members are ranked by their individual score, then considered in order;\n    each is kept if the average *including* it scores at least as well as the\n    average without it. The result is never worse than the best single member\n    on the scoring set, which is what makes greedy preferable to a uniform\n    soup when the members are of uneven quality.\n\n    `score` maps a state dict to a number that is better when larger --\n    macro AUC here. **It must not read the gold 58.** Score on the mined\n    validation fold or on an out-of-fold dump; the whole reason this takes a\n    callable rather than a dataset is to keep that choice at the call site\n    where a reviewer can see it.\n\n    `tolerance` lets a member in when it is within that much of the running\n    best, for the case where the scoring set is too small to resolve the\n    difference and averaging more members is the safer default. At 0.0 this is\n    the paper\'s algorithm exactly.\n\n    Returns the souped state and the names of the members that went into it,\n    in the order they were added.\n    """\n    assert_soupable(states)\n    if names is None:\n        names = [f"member{i}" for i in range(len(states))]\n    if len(names) != len(states):\n        raise ValueError(f"{len(names)} names for {len(states)} states")\n\n    ranked = sorted(zip(names, states), key=lambda pair: score(pair[1]),\n                    reverse=True)\n\n    kept_names: List[str] = [ranked[0][0]]\n    kept_states: List[State] = [ranked[0][1]]\n    best = score(kept_states[0])\n\n    for name, state in ranked[1:]:\n        candidate = average_states(kept_states + [state])\n        value = score(candidate)\n        if value >= best - tolerance:\n            kept_names.append(name)\n            kept_states.append(state)\n            best = max(best, value)\n\n    return average_states(kept_states), kept_names\n\n\ndef rank_average(predictions: Sequence["np.ndarray"]) -> "np.ndarray":\n    """Average ensemble members by within-member rank, not by probability.\n\n    The metric is macro ROC-AUC, which reads only the ordering of studies\n    within a finding. Averaging probabilities lets a member with a compressed\n    or shifted output distribution contribute less than one with a confident\n    one, even when the two rank studies identically -- the mean is sensitive to\n    calibration and AUC is not. Converting each member to ranks first removes\n    that: every member gets one vote per finding, whatever its scale.\n\n    Each member is (N studies, L findings); ranking is per finding, per member,\n    with ties averaged, and the result is scaled to [0, 1] so it can be written\n    into a submission unchanged. Because AUC is invariant to any increasing\n    transform, that scaling costs nothing and keeps the output looking like the\n    probabilities the rest of the pipeline passes around.\n\n    A single member is returned rank-transformed rather than untouched, so that\n    a one-model and a five-model submission are produced by the same code path.\n    """\n    import numpy as np\n\n    if not predictions:\n        raise ValueError("no predictions to average")\n    stacked = np.stack([np.asarray(p, dtype=float) for p in predictions])\n    if stacked.ndim != 3:\n        raise ValueError(\n            f"expected (members, studies, findings), got {stacked.shape}")\n\n    n_studies = stacked.shape[1]\n    if n_studies < 2:\n        # Ranks are meaningless for a single study, and the scaling below\n        # would divide by zero.\n        return stacked.mean(axis=0)\n\n    ranked = np.empty_like(stacked)\n    for member in range(stacked.shape[0]):\n        for finding in range(stacked.shape[2]):\n            column = stacked[member, :, finding]\n            order = np.argsort(column, kind="mergesort")\n            positions = np.empty(n_studies, dtype=float)\n            positions[order] = np.arange(n_studies, dtype=float)\n            # Average the positions of tied values, so a member that predicts\n            # a constant contributes no ordering rather than an arbitrary one.\n            unique, inverse, counts = np.unique(column, return_inverse=True,\n                                                return_counts=True)\n            sums = np.zeros(len(unique))\n            np.add.at(sums, inverse, positions)\n            ranked[member, :, finding] = (sums / counts)[inverse]\n\n    return ranked.mean(axis=0) / (n_studies - 1)\n',
    'src/model/splits.py': '"""Cross-validation splits and the gold holdout.\n\nTwo rules govern every split here, and both exist because the obvious\nalternative produces a number that looks better and means less.\n\n**Split by study, never by series.** A study\'s ~5.5 series image the same knee\nin the same session. A random series split puts near-duplicates on both sides\nand reports an accuracy the model does not have.\n\n**The 58 gold studies never enter training.** They are the only radiologist\nlabels in the dataset and the only honest measurement available. Everything\nelse is trained on mined labels, and a model evaluated against mined labels can\nscore well by reproducing the extractor\'s mistakes. `assert_no_gold_leakage`\nis called by the training entrypoint rather than trusted to discipline.\n"""\n\nfrom typing import Iterator, List, Optional, Sequence, Tuple\n\nimport numpy as np\nimport pandas as pd\n\nfrom src import config\n\nN_FOLDS = 5\nSEED = 0\n\n\ndef load_gold_uids(train_csv=None) -> set:\n    """Study UIDs of the 58 radiologist-labelled studies."""\n    frame = pd.read_csv(train_csv or config.TRAIN_CSV)\n    labelled = frame[frame[config.LABELS[0]].notna()]\n    return set(labelled[config.ID_COL])\n\n\ndef assert_no_gold_leakage(train_uids: Sequence[str], gold_uids: set) -> None:\n    """Fail loudly if a gold study appears in training data.\n\n    Called from the training entrypoint. The failure this guards against is\n    silent -- a leaked gold study inflates the one number being used to decide\n    whether the model works.\n    """\n    leaked = set(train_uids) & gold_uids\n    if leaked:\n        raise ValueError(\n            f"{len(leaked)} gold studies leaked into training: "\n            f"{sorted(leaked)[:3]}..."\n        )\n\n\ndef stratification_key(labels: pd.DataFrame) -> pd.Series:\n    """A coarse key for balancing folds across a 12-label problem.\n\n    Full multi-label stratification over 2^12 combinations would leave most\n    cells empty at 4,349 studies. The count of positive findings captures the\n    axis that matters -- studies with many findings are rare and should not\n    cluster in one fold.\n    """\n    positives = (labels[config.LABELS] > 0.5).sum(axis=1)\n    return positives.clip(upper=4)\n\n\ndef scanner_key(index: pd.DataFrame) -> pd.Series:\n    """A per-study scanner identity, from the header scan\'s index.\n\n    The dataset spans 16-19 sites on five continents, and a CNN reads scanner\n    signature as readily as it reads pathology. A fold split that puts the same\n    scanner on both sides therefore reports an accuracy that does not survive\n    a new site: the plan\'s figure is ~0.053 of inflation from random K-fold,\n    with one measured model showing a 0.136 grouped-vs-random gap.\n\n    Site is not given, so manufacturer + model + field strength stands in for\n    it. That is coarser than a site id -- two hospitals with the same scanner\n    collapse into one group, which makes the split *more* conservative, not\n    less -- and it is what the DICOM headers actually carry.\n\n    Takes the frame `src/data/index.py` writes (one row per instance or\n    series, carrying `StudyInstanceUID`), and returns one key per study,\n    indexed by study UID. A study whose headers name none of the three fields\n    gets "unknown", and those all land in one group together.\n    """\n    fields = [c for c in ("Manufacturer", "ManufacturerModelName",\n                          "MagneticFieldStrength") if c in index.columns]\n    if not fields:\n        raise ValueError(\n            "the index carries none of Manufacturer / ManufacturerModelName / "\n            f"MagneticFieldStrength; it has {list(index.columns)[:8]}")\n\n    # fillna after astype: pandas keeps a missing value missing through\n    # astype(str) rather than rendering it "nan", and the join below would\n    # then fail on a float instead of producing a key.\n    joined = index[fields].astype(str).fillna("nan").agg("|".join, axis=1)\n    joined = joined.where(~joined.str.fullmatch(r"(nan\\|?)+"), "unknown")\n    # The first series scanned decides the study. A study imaged on two\n    # scanners is rare and either group is defensible; taking the first keeps\n    # the assignment deterministic.\n    return joined.groupby(index[config.ID_COL]).first()\n\n\ndef make_folds(labels: pd.DataFrame, n_folds: int = N_FOLDS,\n               seed: int = SEED,\n               groups: Optional[pd.Series] = None) -> pd.DataFrame:\n    """Assign each study to a fold, stratified by positive-finding count.\n\n    Returns the frame with a `fold` column added.\n\n    `groups` maps study UID -> group id (see `scanner_key`). When given, a\n    whole group lands in one fold: the split becomes leave-scanner-out rather\n    than leave-study-out, so validation measures transfer to unseen equipment\n    instead of rewarding memorised scanner signature.\n\n    Grouping costs balance. Groups are sized very unequally -- one scanner may\n    hold a quarter of the corpus -- so groups are assigned largest-first to\n    whichever fold is currently smallest, which is the standard greedy\n    approximation and the best available without splitting a group. Check\n    `summarise()` afterwards: badly imbalanced folds are the price of grouping,\n    and worth knowing before a run rather than after.\n\n    **Changing this re-cuts every fold**, and every reference number on record\n    (fold 1: val 0.872 / gold 0.861 / LB 0.889) belongs to the ungrouped\n    split. Adopt it when the fold ensemble is rebuilt, not in the middle of an\n    experiment that is being compared against those numbers.\n    """\n    frame = labels.copy().reset_index(drop=True)\n    key = stratification_key(frame)\n    rng = np.random.default_rng(seed)\n\n    frame["fold"] = -1\n    if groups is None:\n        for value in key.unique():\n            rows = np.flatnonzero(key.to_numpy() == value)\n            rng.shuffle(rows)\n            frame.loc[rows, "fold"] = np.arange(len(rows)) % n_folds\n    else:\n        mapped = frame[config.ID_COL].map(groups).fillna("unknown")\n        sizes = mapped.value_counts()\n        # Shuffle before sorting so equal-sized groups do not always fall the\n        # same way, then place the largest first into the emptiest fold.\n        order = sizes.sample(frac=1.0, random_state=seed).sort_values(\n            ascending=False, kind="mergesort").index\n        counts = np.zeros(n_folds, dtype=int)\n        for group in order:\n            fold = int(np.argmin(counts))\n            rows = np.flatnonzero(mapped.to_numpy() == group)\n            frame.loc[rows, "fold"] = fold\n            counts[fold] += len(rows)\n\n    if (frame["fold"] < 0).any():\n        raise RuntimeError("some studies were not assigned a fold")\n    return frame\n\n\ndef iter_folds(labels: pd.DataFrame,\n               n_folds: int = N_FOLDS) -> Iterator[Tuple[pd.DataFrame, pd.DataFrame]]:\n    """Yield (train, validation) frames for each fold."""\n    if "fold" not in labels.columns:\n        labels = make_folds(labels, n_folds=n_folds)\n\n    for fold in range(n_folds):\n        validation = labels[labels["fold"] == fold]\n        train = labels[labels["fold"] != fold]\n        yield train.reset_index(drop=True), validation.reset_index(drop=True)\n\n\ndef series_for_studies(manifest: pd.DataFrame,\n                       study_uids: Sequence[str]) -> pd.DataFrame:\n    """All cached series belonging to a set of studies."""\n    wanted = set(study_uids)\n    return manifest[manifest[config.ID_COL].isin(wanted)].reset_index(drop=True)\n\n\ndef summarise(labels: pd.DataFrame, n_folds: int = N_FOLDS) -> pd.DataFrame:\n    """Per-fold positive rates, to confirm the folds are actually comparable."""\n    if "fold" not in labels.columns:\n        labels = make_folds(labels, n_folds=n_folds)\n\n    rows = []\n    for fold in range(n_folds):\n        subset = labels[labels["fold"] == fold]\n        row = {"fold": fold, "studies": len(subset)}\n        for name in config.LABELS:\n            row[name] = float((subset[name] > 0.5).mean())\n        rows.append(row)\n    return pd.DataFrame(rows).set_index("fold")\n',
    'src/labels/__init__.py': ''
}

for path, source in FILES.items():
    target = pathlib.Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source)

sys.path.insert(0, ".")

# Drop any previously imported copy before importing. Writing the files above
# does not affect a module already in sys.modules, so re-running this cell in a
# kernel that has already imported src/ silently keeps the OLD code -- the new
# file sits on disk unused, and the run looks fine until something calls a
# function that only exists in the new version.
for _name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_name]
importlib.invalidate_caches()

from src import config
from src.data import dicom_io, cache, index as index_mod

# Only some notebooks materialise the model package.
if pathlib.Path("src/model/side_classifier.py").exists():
    from src.model import side_classifier
if pathlib.Path("src/model/net.py").exists():
    from src.model import (augment as knee_augment, averaging, conditional,
                           embed, dataset as knee_dataset, net, splits)
# Only the fine-tuning notebook materialises these; lightning imports its
# framework at module scope and fuse pulls in the two extractors.
if pathlib.Path("src/model/inspect.py").exists():
    from src.model import inspect as knee_inspect
if pathlib.Path("src/labels/fuse.py").exists():
    from src.labels import fuse
if pathlib.Path("src/model/lightning.py").exists():
    from src.model import lightning as knee_lightning

print("modules ready:", len(FILES), "files")
print("labels:", len(config.LABELS))
print("cache geometry:", config.CACHE_SLICES, "x", config.CACHE_SIZE)
print("data root:", config.DATA_ROOT)

In [ ]:
import glob
import os

import numpy as np
import torch


def find_inputs(pattern):
    # Datasets mount at /kaggle/input/<name>/, attached notebook output at
    # /kaggle/input/notebooks/<user>/<notebook>/. Skip the competition tree:
    # ~820,000 DICOM files and never what we are looking for.
    found = sorted(glob.glob(f"/kaggle/working/{pattern}"))
    for root in sorted(glob.glob("/kaggle/input/*")):
        if os.path.basename(root) == "competitions":
            continue
        found += sorted(glob.glob(f"{root}/**/{pattern}", recursive=True))
    return found


cache_files = find_inputs("cache_*_series_*.npy")
print("caches:")
for f in cache_files:
    print(" ", f)
if not cache_files:
    raise SystemExit("No cache shards attached.")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## Load the encoder

Downloaded here with internet on, then saved to the working directory. The
submission notebook has no internet and loads the checkpoint from disk.

In [ ]:
encoder = embed.build_encoder(pretrained=True).to(device)
torch.save(encoder.state_dict(), "/kaggle/working/dinov2_s_reg4.pt")

print("model:", embed.MODEL_NAME)
print("params: %.1fM" % (sum(p.numel() for p in encoder.parameters()) / 1e6))
print("embedding dim:", embed.EMBED_DIM)
print("prefix tokens:", encoder.num_prefix_tokens, "(CLS + 4 registers)")

## Embed

Each series contributes 24 slices, stacked 2.5D so every slice carries its
neighbours in the three channels a DINOv2 input expects. Stored as float16:
the precision is irrelevant to a downstream linear layer and it halves both
the file and the load time.

In [ ]:
import time
from tqdm.auto import tqdm

BATCH_SLICES = 256   # slices per forward pass

# 1 = CLS + mean patch (768 dims, 0.9 GB) -- the first model's setting.
# 4 = CLS + 4x4 pooled grid (6,528 dims, 7.6 GB) -- aimed at the focal
#     findings, which averaged 0.695 on gold against 0.865 for regional ones.
# 2 = a middle option at 1,920 dims and 2.3 GB.
embed.POOL_GRID = 4
DIM = embed.embed_dim()
print(f"pool grid {embed.POOL_GRID} -> {DIM} dims/slice "
      f"({24371 * 24 * DIM * 2 / 1e9:.1f} GB over the dataset)")

for path in cache_files:
    stem = os.path.basename(path).replace("cache_", "").replace(".npy", "")
    suffix = "" if embed.POOL_GRID == 1 else f"_g{embed.POOL_GRID}"
    out_path = f"/kaggle/working/embeddings_{stem}{suffix}.npy"
    if os.path.exists(out_path):
        print("skip (exists):", out_path)
        continue

    volumes = np.load(path, mmap_mode="r")
    count, depth = volumes.shape[0], volumes.shape[1]
    output = np.zeros((count, depth, DIM), dtype=np.float16)

    start = time.time()
    buffer, positions = [], []

    def flush():
        if not buffer:
            return
        batch = torch.from_numpy(np.concatenate(buffer)).to(device)
        features = embed.embed_slices(encoder, batch).float().cpu().numpy()
        cursor = 0
        for index, length in positions:
            output[index] = features[cursor:cursor + length].astype(np.float16)
            cursor += length
        buffer.clear()
        positions.clear()

    for i in tqdm(range(count), desc=os.path.basename(path), unit="series"):
        volume = torch.from_numpy(np.asarray(volumes[i]))[None]
        slices = net.stack_2p5d(volume).numpy()
        buffer.append(slices)
        positions.append((i, depth))
        if sum(len(b) for b in buffer) >= BATCH_SLICES:
            flush()
    flush()

    np.save(out_path, output)
    print(f"{out_path}  {output.shape}  "
          f"{os.path.getsize(out_path) / 1e9:.2f} GB  "
          f"{(time.time() - start) / 60:.1f} min")

## Sanity check

Embeddings that are constant across series, or that contain non-finite values,
would train a model that appears to work and predicts nothing. Cheaper to catch
here than after a training run.

In [ ]:
files = sorted(glob.glob("/kaggle/working/embeddings_*.npy"))
for path in files:
    arr = np.load(path, mmap_mode="r")
    sample = np.asarray(arr[:200]).astype(np.float32)
    print(os.path.basename(path), arr.shape)
    print(f"   finite: {np.isfinite(sample).all()}  "
          f"mean {sample.mean():.3f}  std {sample.std():.3f}")
    per_series = sample.reshape(len(sample), -1)
    spread = per_series.std(axis=0).mean()
    print(f"   between-series spread: {spread:.4f}  "
          f"{'OK' if spread > 1e-3 else 'SUSPICIOUS -- embeddings barely vary'}")

## Next

Publish `/kaggle/working` as `knee-embeddings`. Notebook 06 trains the
aggregation stack on these features, and the submission notebook loads
`dinov2_s_reg4.pt` to embed the test series with no internet.